# The 27 July 2026 Chicagoland tornado outbreak: mesoscale environment and radar analysis

Seven tornadoes were confirmed by National Weather Service damage survey across the
Chicago metropolitan area on 27 July 2026 — five EF0 and two EF1, every one of them
between 11:54 am and 1:05 pm CDT (16:54–18:05 UTC). Total surveyed path length was
24.8 miles, the widest damage swath 250 yards, and the peak estimated wind 110 mph.
Seven tornadoes inside seventy minutes is not what late July normally does over
Chicago, and this notebook is an attempt to answer why it did.

The analysis has two halves. The first asks **why the storms were severe**, using the
High-Resolution Rapid Refresh (HRRR) mesoscale environment and the observed
radiosonde network to characterise the airmass, the shear, and the forcing. The second
asks **what actually happened**, using KLOT WSR-88D Level II volumes and the
NWS-surveyed damage polylines to document the storm structure and the tracks.

The short version of the answer is that the thermodynamics were remarkable but not
unprecedented for a Chicago July, and the *kinematics* were the anomaly. Late-July
deep-layer shear over northern Illinois is normally 20–30 kt, which produces
disorganised pulse convection. On 27 July it was 65 kt. That is what turned a very
moist, very unstable airmass into a supercell family instead of an afternoon of
garden-variety thunderstorms.

A second result is worth stating up front because it complicates the simple story:
the significant tornado parameter did not peak during the tornadoes. It was 4.2 at
the tornado hour and climbed to 7.7 by 4 pm CDT — *after* the tornadoes, during a
second round of convection that had gone linear. The environment kept becoming more
tornado-favourable while the storm mode became less so. Storm mode, not environment
alone, decided the hazard.

## Scope and what this notebook does not do

This is the first of three notebooks on the 27–29 July 2026 Chicagoland sequence. It
covers the tornadic event of Monday 27 July. The flooding that followed on 28–29 July,
and the seasonal and climatological context for both, are treated separately.

Every figure in this notebook was produced by the code shown. The figures are embedded
as saved artifacts rather than regenerated at read time, because the input data —
roughly 40 HRRR GRIB2 files, 60 NEXRAD Level II volumes, and a set of web-service
queries — amounts to several gigabytes and several archive endpoints that are not
guaranteed to answer the same way tomorrow. The code is the record of how the figures
were made; re-running it end to end requires re-fetching the data, which the fetch
cells will do.

## 1. Data sources

Every dataset used here is public. The endpoints are given explicitly because several
of them are not the endpoint you would first reach for, and the reasons are documented
in the warnings below.

**HRRR (High-Resolution Rapid Refresh), 3 km CONUS, NOAA/NCEP.** Retrieved with
[Herbie](https://herbie.readthedocs.io/), which resolves to the AWS Open Data mirror
`s3://noaa-hrrr-bdp-pds`. Two products: `sfc` (2D surface and diagnostic fields) and
`prs` (isobaric levels). Cycles 11Z 27 Jul through 05Z 28 Jul, forecast hour F01.

**Observed radiosondes.** Iowa Environmental Mesonet RAOB service,
`https://mesonet.agron.iastate.edu/cgi-bin/request/raob.py`, which redistributes the
NWS/NCEI upper-air archive. Stations KILX (Lincoln IL), KDVN (Quad Cities), KGRB, KMPX,
KDTX, KAPX at 12Z 27 Jul and 00Z 28 Jul.

**Surface observations.** IEM ASOS service,
`https://mesonet.agron.iastate.edu/cgi-bin/request/asos.py`, one-minute-to-hourly
METAR/SPECI for ORD, MDW, DPA, LOT, GYY, PWK, ARR.

**NWS Local Storm Reports.** IEM LSR service,
`https://mesonet.agron.iastate.edu/cgi-bin/request/gis/lsr.py`, WFOs LOT, MKX, IWX,
DVN, ILX.

**Surveyed tornado tracks.** NWS Damage Assessment Toolkit feature service,
`https://services.dat.noaa.gov/arcgis/rest/services/nws_damageassessmenttoolkit/DamageViewer/FeatureServer`,
layer 1 ("Damage Lines").

**NEXRAD Level II, KLOT (Chicago/Romeoville).** Unidata THREDDS radar server,
`https://thredds.ucar.edu/thredds/radarServer/nexrad/level2/IDD`.

**mPING** crowdsourced public reports, `https://mping.nssl.ou.edu/mping/api/v2/reports`
(requires a free API token).

Software: [Py-ART](https://arm-doe.github.io/pyart/) for radar I/O, dealiasing and
gate filtering; [MetPy](https://unidata.github.io/MetPy/) for thermodynamics and
sounding plots; [cmweather](https://cmweather.readthedocs.io/) for the colormaps
(`ChaseSpectral` for reflectivity, `balance` for velocity, `HomeyerRainbow` and
`Theodore16` for scalar mesoscale fields).

### Note on the reported time of the tornadoes

Several news summaries of this event circulate a late-evening tornado window of roughly
22:25–23:35. That is wrong. The NWS Chicago (LOT) public information statement and
damage survey page, and the IEM local storm report archive, both independently place all
seven touchdowns between 16:54 and 18:05 UTC — 11:54 am to 1:05 pm CDT. The surveyed
`starttime`/`endtime` fields in the Damage Assessment Toolkit service agree. Wherever
this notebook and a news account disagree on timing, the survey is authoritative, and
the midday timing matters physically: it puts the tornadoes near local noon, on a
lake-breeze boundary, hours before the environment reached its most tornado-favourable
state.

## 2. Environment and plotting conventions

In [ ]:
import os
import io
import re
import glob
import json
import pickle
import time
import datetime as dt
import warnings

import numpy as np
import pandas as pd
import requests

import matplotlib
matplotlib.use("Agg")
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.dates as mdates
import matplotlib.patheffects as pe
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import pyproj
from scipy.ndimage import gaussian_filter

import metpy.calc as mpcalc
from metpy.units import units
from metpy.plots import SkewT, Hodograph, USCOUNTIES
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

import cmweather  # registers ChaseSpectral, HomeyerRainbow, balance, Theodore16, ...

warnings.filterwarnings("ignore")

# Keep cartopy's Natural Earth downloads inside the working tree.
CARTOPY_DIR = os.path.join(os.getcwd(), "cartopy_data")
os.makedirs(CARTOPY_DIR, exist_ok=True)
cartopy.config["data_dir"] = CARTOPY_DIR
cartopy.config["pre_existing_data_dir"] = CARTOPY_DIR
os.environ["CARTOPY_DATA_DIR"] = CARTOPY_DIR

PC = ccrs.PlateCarree()
GEOD = pyproj.Geod(ellps="WGS84")
CDT = pd.Timedelta(hours=5)          # UTC -> CDT during July
META_GREY = "#888888"

In [ ]:
def apply_figure_style(*, frame="open", font=None, sizes=(8, 7, 6), grid=False):
    """Publication defaults: open frames, small type, left-aligned titles, 300 dpi."""
    if frame not in ("open", "boxed", "none"):
        raise ValueError(f"frame must be 'open'|'boxed'|'none', got {frame!r}")
    base, secondary, tick = sizes
    boxed = (frame == "boxed")
    rc = {
        "font.family": "sans-serif",
        "font.size": base,
        "axes.labelsize": base,
        "axes.titlesize": base,
        "legend.fontsize": secondary,
        "xtick.labelsize": tick,
        "ytick.labelsize": tick,
        "axes.linewidth": 0.6,
        "xtick.direction": "out", "ytick.direction": "out",
        "xtick.major.size": 3, "ytick.major.size": 3,
        "xtick.major.width": 0.6, "ytick.major.width": 0.6,
        "axes.spines.top": boxed, "axes.spines.right": boxed,
        "axes.spines.left": frame != "none", "axes.spines.bottom": frame != "none",
        "axes.grid": bool(grid),
        "legend.frameon": False,
        "figure.dpi": 200,
        "savefig.dpi": 300,
        # 'standard' (not 'tight') so hand-placed figure text keeps its coordinates.
        "savefig.bbox": "standard",
        "axes.titleweight": "normal",
        "axes.titlelocation": "left",
        "lines.linewidth": 1.2,
        "patch.linewidth": 0.6,
        "pdf.fonttype": 42, "ps.fonttype": 42,
    }
    if font:
        rc["font.sans-serif"] = [font, "DejaVu Sans"]
    mpl.rcParams.update(rc)


def panel_letter(ax, letter, dx=-0.18, dy=1.02, fontsize=None):
    if fontsize is None:
        fontsize = plt.rcParams.get("font.size", 8) + 1
    ax.text(dx, dy, letter.lower(), transform=ax.transAxes,
            fontweight="bold", fontsize=fontsize, va="bottom", ha="left")

In [ ]:
# Geography shared by every map in this notebook.

# The seven NWS-confirmed tornadoes, in touchdown order (representative point per track;
# full surveyed polylines are fetched in section 8).
TORNADOES = [("Glen Ellyn",                  41.89, -88.07, "EF0"),
             ("Wheaton",                     41.86, -88.10, "EF0"),
             ("Downers Grove",               41.79, -88.02, "EF0"),
             ("Romeoville",                  41.63, -88.08, "EF0"),
             ("Orland Park",                 41.65, -87.90, "EF0"),
             ("Country Club Hills\u2013Glenwood", 41.58, -87.67, "EF1"),
             ("Lynwood IL\u2013Munster IN",       41.53, -87.53, "EF1")]

CITIES = [("Chicago", 41.878, -87.630), ("Rockford", 42.271, -89.094),
          ("Joliet", 41.525, -88.081), ("Gary", 41.593, -87.346),
          ("Milwaukee", 43.039, -87.906), ("Aurora", 41.760, -88.320)]

# Suburbs added on the close-in track and radar maps.
SUBURBS = CITIES + [("Wheaton", 41.866, -88.107), ("Downers Grove", 41.809, -88.011),
                    ("Orland Park", 41.630, -87.854), ("Country Club Hills", 41.568, -87.720),
                    ("Romeoville", 41.647, -88.090), ("Lynwood", 41.526, -87.539),
                    ("Glen Ellyn", 41.878, -88.067)]

# Analysis points. Glen Ellyn / Wheaton is the first-touchdown corridor and is used for
# every single-column diagnostic in this notebook.
SITES = {"Chicago O'Hare (ORD)":              (41.9803, -87.9090),
         "Glen Ellyn / Wheaton":              (41.8800, -88.0800),
         "Midway (MDW)":                      (41.7860, -87.7524),
         "Hazel Crest / Country Club Hills":  (41.5800, -87.6700),
         "Argonne (Lemont)":                  (41.7183, -87.9786)}
GL = SITES["Glen Ellyn / Wheaton"]
ORD = SITES["Chicago O'Hare (ORD)"]

# Metro map domain, and the wider domain for synoptic panels.
DOM = dict(lon_min=-89.6, lon_max=-85.8, lat_min=40.6, lat_max=43.1)
DOM_SYN = dict(lon_min=-94.5, lon_max=-83.5, lat_min=39.0, lat_max=44.8)

# Event timing (CDT). The tornado window used for shading throughout.
TORN_T0 = pd.Timestamp("2026-07-27 11:54")
TORN_T1 = pd.Timestamp("2026-07-27 12:59")

## 3. The observed airmass: radiosondes bracketing the event

Before trusting any model field, it is worth asking what the observing network actually
measured. Chicago has no radiosonde site; the nearest launches are KILX (Lincoln,
Illinois, ~200 km southwest) and KDVN (Quad Cities, ~250 km west). At 12Z on 27 July
both sampled the airmass that would be advected into the metro over the following five
hours, so they are the observational anchor for the model environment.

The KILX 12Z sounding gives **MUCAPE 5965 J kg⁻¹** with **43 kt of deep-layer shear**.
Nearly 6000 J kg⁻¹ of most-unstable CAPE at 7 am local is a genuinely extreme
thermodynamic profile — the airmass was loaded before the sun had done much work on it.
The 43 kt of shear at KILX is already well above the July norm for the region, and the
HRRR analysis will show it strengthening further as the flow reached Chicago.

In [ ]:
def iem_raob(stn, when):
    """One radiosonde from the IEM RAOB archive as a DataFrame."""
    u = ("https://mesonet.agron.iastate.edu/cgi-bin/request/raob.py?"
         f"station={stn}&sts={when:%Y-%m-%dT%H:%M}Z"
         f"&ets={when + pd.Timedelta(minutes=1):%Y-%m-%dT%H:%M}Z&fmt=csv")
    return pd.read_csv(io.StringIO(requests.get(u, timeout=120).text), na_values=["M"])


STNS = ["KILX", "KDVN", "KGRB", "KMPX", "KDTX", "KAPX"]
TIMES = [pd.Timestamp("2026-07-27 12:00"), pd.Timestamp("2026-07-28 00:00")]

os.makedirs("soundings", exist_ok=True)
raob = {}
for s in STNS:
    for w in TIMES:
        try:
            df = (iem_raob(s, w).dropna(subset=["pressure_mb"])
                  .sort_values("pressure_mb", ascending=False))
            if len(df) > 20:
                tag = f"{s}_{w:%d%HZ}"
                raob[tag] = df
                df.to_csv(f"soundings/{tag}.csv", index=False)
        except Exception as e:
            print(s, w, "ERR", type(e).__name__, str(e)[:80])

print(f"{len(raob)} soundings retrieved:", ", ".join(sorted(raob)))

In [ ]:
def prep_thermo(df):
    """Monotonic, de-duplicated p/T/Td profile."""
    d = df.dropna(subset=["pressure_mb", "tmpc", "dwpc"]).copy()
    return (d[d.pressure_mb > 0].drop_duplicates(subset="pressure_mb")
            .sort_values("pressure_mb", ascending=False))


def prep_wind(df):
    w = df.dropna(subset=["pressure_mb", "height_m", "drct", "speed_kts"]).copy()
    return w[w.pressure_mb > 0].drop_duplicates(subset="height_m").sort_values("height_m")


def raob_params(df):
    """Derived severe-weather parameters for one observed sounding.

    Every block is guarded: real radiosondes have missing wind levels, balloon
    terminations below 6 km and duplicate pressures, and a single failure should not
    cost the whole profile.
    """
    d, w = prep_thermo(df), prep_wind(df)
    p = d.pressure_mb.values * units.hPa
    T = d.tmpc.values * units.degC
    Td = d.dwpc.values * units.degC
    o = {"n_thermo": len(d), "n_wind": len(w)}

    for lab, fn in [("SB", mpcalc.surface_based_cape_cin),
                    ("MU", mpcalc.most_unstable_cape_cin)]:
        try:
            c, ci = fn(p, T, Td)
            o[lab + "CAPE"], o[lab + "CIN"] = c.m, ci.m
        except Exception:
            pass
    try:
        c, ci = mpcalc.mixed_layer_cape_cin(p, T, Td, depth=100 * units.hPa)
        o["MLCAPE"], o["MLCIN"] = c.m, ci.m
    except Exception:
        pass
    try:
        lp, _ = mpcalc.lcl(p[0], T[0], Td[0])
        o["LCL_hPa"] = lp.m
        o["LCL_m"] = float(np.interp(-lp.m, -d.pressure_mb.values, d.height_m.values)
                           - d.height_m.values[0])
    except Exception:
        pass
    try:
        o["PWAT_mm"] = mpcalc.precipitable_water(p, Td).to("mm").m
    except Exception:
        pass
    try:
        zz = d.height_m.values - d.height_m.values[0]
        o["LR_0_3km"] = float(-(np.interp(3000, zz, d.tmpc.values) - d.tmpc.values[0]) / 3.0)
        t7 = np.interp(-700, -d.pressure_mb.values, d.tmpc.values)
        t5 = np.interp(-500, -d.pressure_mb.values, d.tmpc.values)
        z7 = np.interp(-700, -d.pressure_mb.values, d.height_m.values)
        z5 = np.interp(-500, -d.pressure_mb.values, d.height_m.values)
        o["LR_700_500"] = float(-(t5 - t7) / ((z5 - z7) / 1000.0))
    except Exception:
        pass

    if len(w) >= 8:
        zw = w.height_m.values * units.m
        pw = w.pressure_mb.values * units.hPa
        uu, vv = mpcalc.wind_components(w.speed_kts.values * units.knots,
                                        w.drct.values * units.deg)
        agl_top = (w.height_m.values - w.height_m.values[0]).max()
        for top, lab in [(1000, "01"), (3000, "03"), (6000, "06")]:
            if agl_top >= top:
                try:
                    bs = mpcalc.bulk_shear(pw, uu, vv, height=zw, depth=top * units.m)
                    o[f"SHR{lab}_kt"] = float(mpcalc.wind_speed(*bs).to("knots").m)
                except Exception:
                    pass
        try:
            rm, lm, mn = mpcalc.bunkers_storm_motion(pw, uu, vv, zw)
            o["BunkR_kt"] = float(mpcalc.wind_speed(*rm).to("knots").m)
            o["BunkR_dir"] = float(mpcalc.wind_direction(*rm).m)
            for top, lab in [(1000, "01"), (3000, "03")]:
                s = mpcalc.storm_relative_helicity(zw, uu, vv, depth=top * units.m,
                                                   storm_u=rm[0], storm_v=rm[1])
                o[f"SRH{lab}"] = float(s[2].m)
        except Exception:
            pass
    return o


P = pd.DataFrame({t: raob_params(df) for t, df in raob.items()}).T
P.index.name = "sounding"
P.to_csv("observed_sounding_params.csv")
print(P[["MUCAPE", "MLCAPE", "MLCIN", "PWAT_mm", "SHR06_kt", "SRH01"]].round(0).to_string())

In [ ]:
PANELS_RAOB = [("KILX_2712Z", "Lincoln IL  27 Jul 12 UTC", "Upstream/inflow air, 5 h before tornadoes"),
               ("KDVN_2712Z", "Quad Cities  27 Jul 12 UTC", "Upstream, western flank"),
               ("KILX_2800Z", "Lincoln IL  28 Jul 00 UTC", "Post-event, evening"),
               ("KDVN_2800Z", "Quad Cities  28 Jul 00 UTC", "Peak-heating airmass")]

SND_LEGEND = [Line2D([], [], color="#c0392b", lw=1.8, label="Temperature"),
              Line2D([], [], color="#1a7a3d", lw=1.8, label="Dewpoint"),
              Line2D([], [], color="k", lw=1.1, ls="--", label="Surface parcel path"),
              Patch(fc="#e8743b", alpha=0.22, label="CAPE (positive area)"),
              Patch(fc="#5b6b8c", alpha=0.25, label="CIN (inhibition)")]


def skew_panel(fig, subplot_spec, p, T, Td, u, v, pw, title, sub, stats, letter,
               hodo_range=60):
    """One skew-T / log-p panel with a shaded parcel path and an inset hodograph."""
    skew = SkewT(fig, rotation=45, subplot=subplot_spec)
    skew.plot(p, T, "#c0392b", lw=1.8)
    skew.plot(p, Td, "#1a7a3d", lw=1.8)
    prof = mpcalc.parcel_profile(p, T[0], Td[0]).to("degC")
    skew.plot(p, prof, "k", lw=1.1, ls="--")
    skew.shade_cape(p, T, prof, alpha=0.22, color="#e8743b")
    skew.shade_cin(p, T, prof, alpha=0.25, color="#5b6b8c")
    skew.plot_barbs(pw, u, v, length=5.5, xloc=1.06)
    skew.ax.set_ylim(1020, 100)
    skew.ax.set_xlim(-32, 42)
    skew.plot_dry_adiabats(lw=0.4, alpha=0.35)
    skew.plot_moist_adiabats(lw=0.4, alpha=0.35)
    skew.plot_mixing_lines(lw=0.4, alpha=0.35)
    skew.ax.set_xlabel("Temperature (\u00b0C)")
    skew.ax.set_ylabel("Pressure (hPa)")
    skew.ax.set_title(f"{title}\n{sub}", loc="left")
    skew.ax.text(0.02, 0.02, stats, transform=skew.ax.transAxes, fontsize=7.4,
                 va="bottom", bbox=dict(fc="white", ec="0.7", lw=0.5, alpha=0.92,
                                        boxstyle="round,pad=0.4"))
    panel_letter(skew.ax, letter, dx=-0.12, dy=1.12)
    return skew


def hodo_inset(ax, u, v, z_agl, component_range=60):
    """0-9 km hodograph inset, coloured by height. Axis tick labels are suppressed and
    the explanation goes in the xlabel: an inset title collides with the skew-T."""
    axh = inset_axes(ax, width="34%", height="34%", loc="upper right", borderpad=0.6)
    h = Hodograph(axh, component_range=component_range)
    h.add_grid(increment=20, lw=0.4, alpha=0.5)
    h.plot_colormapped(u, v, z_agl, cmap=plt.get_cmap("plasma"), lw=1.7)
    axh.set_xticklabels([])
    axh.set_yticklabels([])
    axh.set_xlabel("0\u20139 km hodograph (kt)\npurple\u2192yellow = 0\u21929 km",
                   fontsize=6.2, labelpad=1.5)
    return axh


plt.close("all")
apply_figure_style(sizes=(9, 8, 7))
fig = plt.figure(figsize=(13.5, 12.6))
gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.36, wspace=0.20)

for k, (tag, title, sub) in enumerate(PANELS_RAOB):
    d, w = prep_thermo(raob[tag]), prep_wind(raob[tag])
    p = d.pressure_mb.values * units.hPa
    T = d.tmpc.values * units.degC
    Td = d.dwpc.values * units.degC

    # Barbs: thin to ~28 levels below 12 km so they stay legible.
    ww = w[w.height_m - w.height_m.iloc[0] <= 12000]
    ws = ww.iloc[::max(1, len(ww) // 28)]
    bu, bv = mpcalc.wind_components(ws.speed_kts.values * units.knots,
                                    ws.drct.values * units.deg)

    pr = P.loc[tag]
    stats = (f"MUCAPE {pr.MUCAPE:.0f} J kg$^{{-1}}$\nMLCAPE {pr.MLCAPE:.0f} J kg$^{{-1}}$\n"
             f"MLCIN {pr.MLCIN:.0f} J kg$^{{-1}}$\nPWAT {pr.PWAT_mm:.0f} mm\n"
             f"0\u20136 km shear {pr.SHR06_kt:.0f} kt\n"
             f"0\u20131 km SRH {pr.SRH01:.0f} m$^2$ s$^{{-2}}$")

    skew = skew_panel(fig, gs[k], p, T, Td, bu, bv,
                      ws.pressure_mb.values * units.hPa, title, sub, stats, "abcd"[k])

    hh = w[w.height_m - w.height_m.iloc[0] <= 9000]
    hu, hv = mpcalc.wind_components(hh.speed_kts.values * units.knots,
                                    hh.drct.values * units.deg)
    hodo_inset(skew.ax, hu, hv,
               (hh.height_m.values - hh.height_m.values[0]) * units.m)

fig.legend(handles=SND_LEGEND, loc="lower center", ncol=5, frameon=False,
           bbox_to_anchor=(0.5, 0.05), fontsize=8.5)
fig.suptitle("Observed radiosondes bracketing the 27 July 2026 Chicagoland "
             "severe-weather event", x=0.09, ha="left", fontsize=12, y=0.955)
fig.savefig("observed_soundings.png", dpi=200, bbox_inches="tight")

![Observed radiosondes bracketing the event. (a) KILX and (b) KDVN at 12Z 27 July sample the inflow airmass five hours before the first touchdown; (c) and (d) are the 00Z 28 July launches. Insets are 0-9 km hodographs coloured by height.](01_tornadic_event_27jul2026_outputs/observed_soundings.png)

*Observed radiosondes bracketing the event. (a) KILX and (b) KDVN at 12Z 27 July sample the inflow airmass five hours before the first touchdown; (c) and (d) are the 00Z 28 July launches. Insets are 0-9 km hodographs coloured by height.*

The 12Z KILX profile (panel a) is the one to read. The dewpoint curve is nearly
saturated through the lowest 150 hPa, mid-level lapse rates are steep, and the parcel
path leaves the environment curve well to the right through most of the troposphere —
hence 5965 J kg⁻¹ of most-unstable CAPE. The hodograph inset shows appreciable
low-level curvature, with 43 kt of 0–6 km shear.

The caveat on all of this is spatial. KILX is 200 km from Glen Ellyn and the launch is
five hours before the event. These soundings establish that the airmass moving toward
Chicago was extreme, not that the Chicago column looked like this at 17Z. For that we
need the model, which is the next section.

## 4. HRRR retrieval

Two harvests are needed: a surface/diagnostic set at hourly resolution through the whole
event, and an isobaric set at four selected times for soundings and synoptic maps.
Both go through Herbie.

> **Warning — use `fxx=1`, not `fxx=0`.**
>
> It is tempting to take the F00 analysis as "the closest thing to truth". For
> convective diagnostics this is a mistake. The HRRR analysis has no spun-up
> convection: vertical velocity is essentially zero, and the *max-in-previous-hour*
> diagnostics that convective analysis depends on — `MXUPHL` (updraft helicity),
> `MAXUVV` (maximum updraft), `APCP` (accumulated precipitation) — do not exist in the
> F00 file at all, because there is no previous hour.
>
> Verified for this case: 700 hPa ascent in the F01 field is roughly **38× stronger**
> than in F00 over the convective corridor. The fix is to take F01 from the cycle one
> hour earlier, so the valid time is what you asked for and the model has had an hour
> to build storms. Every hourly field in this notebook is `Herbie(valid - 1h, fxx=1)`.

> **Warning — storm-relative helicity layers collide.**
>
> In HRRR GRIB2, the 0–1 km and 0–3 km storm-relative helicity fields share *both* the
> variable short name (`HLCY`) and the `typeOfLevel`
> (`heightAboveGroundLayer`). They differ only in the level value. A harvest routine
> keyed on `(varname, typeOfLevel)` will silently keep whichever one it read last, and
> you will not get an error — you will get a plausible-looking SRH field that is the
> wrong layer.
>
> Two defences are used below: the harvest key includes the numeric level, and the two
> layers are requested in separate `H.xarray()` calls with explicit level search
> strings. Both are cheap. Getting this wrong quietly corrupts every composite index
> that uses SRH, which is most of them.

In [ ]:
"""fetch_hrrr.py -- HRRR surface and diagnostic fields over Chicagoland.

Valid times 12Z 27 Jul through 06Z 28 Jul, taken as F01 from the preceding cycle.
"""
from herbie import Herbie

SAVE = os.path.join(os.getcwd(), "herbie_data")
os.makedirs(SAVE, exist_ok=True)

VALID = pd.date_range("2026-07-27 12:00", "2026-07-28 06:00", freq="1h", tz=None)
LAT0, LAT1, LON0, LON1 = 39.8, 43.8, -92.0, -85.0     # sub-domain retained for maps

# (group name, GRIB search string, {(varname, typeOfLevel[, level]): output key})
# Note the SRH group: keyed on the numeric level, and requested on its own so the
# 0-1 km and 0-3 km layers cannot overwrite one another.
GROUPS = [
    ("bulk", ":(CAPE|CIN):(surface|180-0 mb above ground):|:PWAT:|:MSLMA:|:GUST:surface:"
             "|:HPBL:surface:|:(TMP|DPT):2 m above ground:|:[UV]GRD:10 m above ground:"
             "|:APCP:surface:0-1 hour acc fcst:|:HGT:level of adiabatic condensation from sfc:"
             "|:(REFC|VIL|LTNG):|:RETOP:cloud top:",
     {("cape", "surface"): "cape_sfc", ("cin", "surface"): "cin_sfc",
      ("cape", "pressureFromGroundLayer"): "cape_ml",
      ("cin", "pressureFromGroundLayer"): "cin_ml",
      ("pwat", "atmosphereSingleLayer"): "pwat", ("mslma", "meanSea"): "mslp",
      ("gust", "surface"): "gust", ("blh", "surface"): "pblh",
      ("t2m", "heightAboveGround"): "t2m", ("d2m", "heightAboveGround"): "d2m",
      ("u10", "heightAboveGround"): "u10", ("v10", "heightAboveGround"): "v10",
      ("tp", "surface"): "apcp1h", ("gh", "adiabaticCondensation"): "lcl_agl",
      ("refc", "atmosphere"): "refc", ("veril", "atmosphere"): "vil",
      ("ltng", "atmosphere"): "ltng", ("unknown", "cloudTop"): "echotop"}),
    ("cape03", ":CAPE:0-3000 m above ground:",
     {("cape", "heightAboveGroundLayer"): "cape_03km"}),
    ("srh",    ":HLCY:", {("hlcy", "heightAboveGroundLayer", 3000.0): "srh_03km",
                          ("hlcy", "heightAboveGroundLayer", 1000.0): "srh_01km"}),
    ("shr6",   ":(VUCSH|VVCSH):0-6000 m above ground:",
     {("vucsh", "heightAboveGroundLayer"): "ushr_06km",
      ("vvcsh", "heightAboveGroundLayer"): "vshr_06km"}),
    ("shr1",   ":(VUCSH|VVCSH):0-1000 m above ground:",
     {("vucsh", "heightAboveGroundLayer"): "ushr_01km",
      ("vvcsh", "heightAboveGroundLayer"): "vshr_01km"}),
    ("stm",    ":[UV]STM:0-6000 m above ground:",
     {("ustm", "heightAboveGroundLayer"): "ustm", ("vstm", "heightAboveGroundLayer"): "vstm"}),
    ("uh25",   ":MXUPHL:2000-0 m above ground:",
     {("unknown", "heightAboveGroundLayer"): "uh_02km"}),
    ("uh52",   ":MXUPHL:5000-2000 m above ground:",
     {("unknown", "heightAboveGroundLayer"): "uh_25km"}),
    ("uvv",    ":MAXUVV:", {("unknown", "pressureFromGroundLayer"): "maxuvv"}),
]


def harvest(ds, mapping, out):
    """Copy wanted variables out of a Herbie dataset, keyed on (name, level type, level)."""
    for v in ds.data_vars:
        tol = ds[v].attrs.get("GRIB_typeOfLevel", "?")
        lev = float(ds[tol].values) if tol in ds.coords and ds[tol].size == 1 else None
        key = mapping.get((v, tol, lev)) or mapping.get((v, tol))
        if key:
            out[key] = ds[v].values.astype("float32")


rows, maps, geo = [], {}, {}
for vt in VALID:
    cycle = vt - pd.Timedelta(hours=1)                 # F01 valid at vt
    H = Herbie(cycle, model="hrrr", product="sfc", fxx=1, save_dir=SAVE, verbose=False)
    flds = {}
    for name, search, mapping in GROUPS:
        try:
            res = H.xarray(search, remove_grib=False)
            for ds in (res if isinstance(res, list) else [res]):
                harvest(ds, mapping, flds)
                if "lat" not in geo:
                    la = ds.latitude.values
                    lo = np.where(ds.longitude.values > 180,
                                  ds.longitude.values - 360, ds.longitude.values)
                    geo["lat"], geo["lon"] = la, lo
        except Exception as e:
            print(f"  {vt:%H}Z {name} FAIL {type(e).__name__} {str(e)[:90]}", flush=True)

    if "lat" in geo and "clipped" not in geo:
        la, lo = geo["lat"], geo["lon"]
        m = (la >= LAT0) & (la <= LAT1) & (lo >= LON0) & (lo <= LON1)
        yy, xx = np.where(m)
        geo["j0"], geo["j1"] = yy.min(), yy.max() + 1
        geo["i0"], geo["i1"] = xx.min(), xx.max() + 1
        geo["clipped"] = True
    j0, j1, i0, i1 = geo["j0"], geo["j1"], geo["i0"], geo["i1"]
    for k, a in flds.items():
        maps.setdefault(k, []).append(a[j0:j1, i0:i1])
    print(f"{vt:%Y-%m-%d %HZ} ok  nfields={len(flds)}", flush=True)

out = dict(valid=VALID, maps={k: np.stack(v) for k, v in maps.items()},
           lat=geo["lat"][geo["j0"]:geo["j1"], geo["i0"]:geo["i1"]],
           lon=geo["lon"][geo["j0"]:geo["j1"], geo["i0"]:geo["i1"]])
pickle.dump(out, open("hrrr_chicago_20260727.pkl", "wb"))
print("saved", len(out["maps"]), "fields on", out["lat"].shape, "grid")

In [ ]:
"""fetch_prs.py -- HRRR isobaric fields for point soundings and synoptic maps.

Four valid times only, on a wider domain than the surface harvest. These are taken as
F00 analyses because they are used for the thermodynamic and kinematic *state*; the one
field that genuinely needs spun-up convection -- 700 hPa vertical velocity -- is
re-fetched as F01 in section 7.
"""
from herbie import Herbie

TIMES_PRS = [pd.Timestamp("2026-07-27 12:00"), pd.Timestamp("2026-07-27 15:00"),
             pd.Timestamp("2026-07-27 17:00"), pd.Timestamp("2026-07-27 21:00")]
LAT0, LAT1, LON0, LON1 = 38.5, 45.0, -95.0, -83.0

out = {}
for vt in TIMES_PRS:
    H = Herbie(vt, model="hrrr", product="prs", fxx=0, save_dir=SAVE, verbose=False)
    got = {}
    for key, search in [("gh", r":HGT:\d+ mb:"), ("t", r":TMP:\d+ mb:"),
                        ("dpt", r":DPT:\d+ mb:"), ("u", r":UGRD:\d+ mb:"),
                        ("v", r":VGRD:\d+ mb:"), ("w", r":VVEL:\d+ mb:"),
                        ("absv", r":ABSV:\d+ mb:"), ("q", r":SPFH:\d+ mb:")]:
        try:
            res = H.xarray(search, remove_grib=False)
            ds = res[0] if isinstance(res, list) else res
            got[key] = ds[list(ds.data_vars)[0]]
            if "lat" not in out:
                la = ds.latitude.values
                lo = np.where(ds.longitude.values > 180,
                              ds.longitude.values - 360, ds.longitude.values)
                yy, xx = np.where((la >= LAT0) & (la <= LAT1) & (lo >= LON0) & (lo <= LON1))
                out["j0"], out["j1"] = yy.min(), yy.max() + 1
                out["i0"], out["i1"] = xx.min(), xx.max() + 1
                out["lat"] = la[out["j0"]:out["j1"], out["i0"]:out["i1"]]
                out["lon"] = lo[out["j0"]:out["j1"], out["i0"]:out["i1"]]
                out["levels"] = ds.isobaricInhPa.values
        except Exception as e:
            print(f"  {vt:%H}Z {key} FAIL {type(e).__name__} {str(e)[:80]}", flush=True)
    j0, j1, i0, i1 = out["j0"], out["j1"], out["i0"], out["i1"]
    out[str(vt)] = {k: da.values[:, j0:j1, i0:i1].astype("float32") for k, da in got.items()}
    print(f"{vt:%HZ} ok {list(got)} levels={len(out['levels'])}", flush=True)

pickle.dump(out, open("hrrr_prs.pkl", "wb"))
print("saved. levels", out["levels"][0], "->", out["levels"][-1], "grid", out["lat"].shape)

## 5. Derived fields and composite indices

The GRIB harvest gives raw SI fields. Everything plotted below is derived from them in
one place, so that unit conversions live in exactly one cell and cannot drift between
figures.

Two composite indices are computed: the supercell composite parameter (SCP) and the
significant tornado parameter (STP), both in the fixed-layer formulation. These are
recomputed from HRRR fields rather than taken from a model diagnostic, so the exact
numbers will differ slightly from SPC's operational mesoanalysis, which uses different
input layers and an effective-inflow-layer definition. They are used here for spatial
pattern and time evolution, not as calibrated probabilities.

In [ ]:
D = pickle.load(open("hrrr_chicago_20260727.pkl", "rb"))
M, LAT, LON = D["maps"], D["lat"], D["lon"]
VT = pd.DatetimeIndex(D["valid"])

# --- unit conversions -------------------------------------------------------
M["shr06_kt"] = np.hypot(M["ushr_06km"], M["vshr_06km"]) * 1.94384
M["shr01_kt"] = np.hypot(M["ushr_01km"], M["vshr_01km"]) * 1.94384
M["t2f"] = M["t2m"] * 9 / 5 - 459.67
M["d2f"] = M["d2m"] * 9 / 5 - 459.67
M["gust_mph"] = M["gust"] * 2.23694
M["pwat_in"] = M["pwat"] / 25.4
M["apcp_in"] = M["apcp1h"] / 25.4
M["mslp_hpa"] = M["mslp"] / 100.0

# --- supercell composite parameter -----------------------------------------
# SCP = (MUCAPE/1000) * (SRH_0-3km/50) * (shear term) * (CIN term)
mucape = np.maximum(M["cape_ml"], M["cape_sfc"])
eshr = np.clip(M["shr06_kt"] / 1.94384, 0, None)          # back to m/s
s_term = np.clip(eshr / 20.0, 0, 1.5)
s_term[eshr < 10] = 0.0
c_term = np.where(M["cin_ml"] > -40, 1.0, -40.0 / np.minimum(M["cin_ml"], -1e-6))
M["scp"] = (mucape / 1000.0) * (M["srh_03km"] / 50.0) * s_term * c_term

# --- significant tornado parameter (fixed layer) ---------------------------
# STP = (MLCAPE/1500) * (LCL term) * (SRH_0-1km/150) * (shear term) * (CIN term)
lcl_term = np.clip((2000.0 - M["lcl_agl"]) / 1000.0, 0, 1.0)
lcl_term[M["lcl_agl"] < 1000] = 1.0
cin_term = np.clip((200.0 + M["cin_ml"]) / 150.0, 0, 1.0)
cin_term[M["cin_ml"] > -50] = 1.0
sh_term = np.clip(eshr / 20.0, 0, 1.5)
sh_term[eshr < 12.5] = 0.0
M["stp"] = (M["cape_ml"] / 1500.0) * lcl_term * (M["srh_01km"] / 150.0) * sh_term * cin_term

pickle.dump(dict(maps=M, lat=LAT, lon=LON, valid=VT),
            open("hrrr_chicago_derived.pkl", "wb"))

it = list(VT).index(pd.Timestamp("2026-07-27 17:00"))   # the tornado hour
print("fields:", len(M), " tornado-hour index:", it, VT[it])

In [ ]:
def at(field, lat, lon, t=None):
    """Nearest-gridpoint time series (or single value) from the HRRR sub-domain."""
    j, i = np.unravel_index(np.argmin((LAT - lat) ** 2 + (LON - lon) ** 2), LAT.shape)
    return M[field][:, j, i] if t is None else M[field][t, j, i]


# The tornado-hour environment at the first-touchdown corridor, as quoted in the text.
env = {"MLCAPE (J/kg)":      at("cape_ml", *GL, it),
       "0-6 km shear (kt)":  at("shr06_kt", *GL, it),
       "0-1 km SRH (m2/s2)": at("srh_01km", *GL, it),
       "LCL (m AGL)":        at("lcl_agl", *GL, it),
       "PWAT (in)":          at("pwat_in", *GL, it),
       "2 m dewpoint (F)":   at("d2f", *GL, it),
       "STP":                at("stp", *GL, it)}
for k, v in env.items():
    print(f"  {k:22s} {v:8.1f}")
print(f"  {'domain max MLCAPE':22s} {M['cape_ml'][it].max():8.1f}")

At 17Z over Glen Ellyn the HRRR gives **MLCAPE 4350 J kg⁻¹** (domain maximum 4820),
**0–6 km bulk shear 65 kt**, **0–1 km SRH 144 m² s⁻²**, an **LCL of 390 m AGL**,
**PWAT 2.29 in**, and a **2 m dewpoint of 80 °F**. Read as a set, that is a
high-CAPE, low-LCL, strongly-sheared profile: the classic ingredients.

The single number that separates this day from a normal Chicago late July is the shear.
Climatologically, 0–6 km bulk shear over northern Illinois in late July runs 20–30 kt,
which is why summer convection here is usually a field of disorganised pulse cells that
produce brief microbursts and little else. At 65 kt, updrafts and downdrafts separate,
mesocyclones survive, and the same storm can produce tornado after tornado for over an
hour. The moisture and instability made the storms strong; the shear made them
organised.

## 6. The mesoscale environment at the tornado hour

Six fields at 17Z, all on the same metro domain, with the seven confirmed tornado
locations overplotted so the spatial correspondence can be judged directly rather than
asserted.

In [ ]:
def basemap(ax, label_cities=False, domain=DOM, counties="5m"):
    ax.set_extent([domain["lon_min"], domain["lon_max"],
                   domain["lat_min"], domain["lat_max"]], crs=PC)
    ax.add_feature(USCOUNTIES.with_scale(counties), edgecolor="0.70", lw=0.2, zorder=2)
    ax.add_feature(cfeature.STATES.with_scale("10m"), edgecolor="0.1", lw=0.9, zorder=6)
    ax.add_feature(cfeature.LAKES.with_scale("10m"), fc="none", ec="0.05", lw=1.0, zorder=6)
    if label_cities:
        for nm, la, lo in CITIES:
            ax.plot(lo, la, "o", ms=2.6, color="k", transform=PC, zorder=8)
            ax.text(lo + 0.05, la + 0.035, nm, fontsize=6.4, color="k", transform=PC,
                    zorder=8, path_effects=[pe.withStroke(linewidth=1.6, foreground="white")])


def torn_marks(ax, ms=45, lw=1.1):
    for nm, la, lo, ef in TORNADOES:
        ax.scatter(lo, la, marker="v", s=ms, c="none", edgecolors="k", lw=lw,
                   transform=PC, zorder=9)


# (field, colorbar label, contour levels, cmweather colormap, extend, panel subtitle)
PANELS_ENV = [
    ("cape_ml", "MLCAPE (J kg$^{-1}$)", np.arange(0, 5001, 250), "HomeyerRainbow", "max",
     "Extreme instability \u2014 over 4000 J kg$^{-1}$ across the metro"),
    ("d2f", "2 m dewpoint (\u00b0F)", np.arange(62, 84.1, 1), "BlueBrown10_r", "both",
     "Tropical low-level moisture, dewpoints near 80 \u00b0F"),
    ("shr06_kt", "0\u20136 km bulk shear (kt)", np.arange(15, 71, 2.5), "LangRainbow12", "both",
     "Deep-layer shear above 50 kt \u2014 supercell range"),
    ("srh_01km", "0\u20131 km storm-relative helicity (m$^2$ s$^{-2}$)",
     np.arange(0, 401, 20), "Theodore16", "both",
     "Low-level rotation maximised over DuPage and Cook"),
    ("stp", "Significant tornado parameter", np.arange(0, 8.1, 0.4), "ChaseSpectral", "max",
     "Tornado parameter above 3 along the damage corridor"),
    ("refc", "Simulated composite reflectivity (dBZ)", np.arange(5, 71, 2.5),
     "ChaseSpectral", "max", "HRRR reproduces the discrete supercell corridor"),
]

plt.close("all")
apply_figure_style(sizes=(9.5, 8.5, 7.5))
fig = plt.figure(figsize=(15.5, 9.6))
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.20, wspace=0.09)

for k, (f, label, levs, cm, ext, sub) in enumerate(PANELS_ENV):
    ax = fig.add_subplot(gs[k], projection=PC)
    basemap(ax, label_cities=(k in (0, 5)))
    Z = M[f][it].copy()
    if f in ("cape_ml", "srh_01km", "shr06_kt", "stp"):
        Z = gaussian_filter(Z, 0.8)            # 3 km grid noise, not signal
    if f in ("stp", "srh_01km"):
        Z = np.clip(Z, 0, None)                # negative values are not meaningful here
    if f == "refc":
        Z = np.where(Z < 5, np.nan, Z)         # blank clear air
    cf = ax.contourf(LON, LAT, Z, levels=levs, cmap=cm, extend=ext, transform=PC, zorder=3)
    cb = fig.colorbar(cf, ax=ax, orientation="horizontal", pad=0.03, aspect=32, shrink=0.94)
    cb.set_label(label, fontsize=8)
    cb.ax.tick_params(labelsize=7)
    torn_marks(ax)
    ax.set_title(sub, loc="left", fontsize=8.8, pad=14)
    panel_letter(ax, "abcdef"[k], dx=-0.012, dy=1.145)

fig.suptitle(f"HRRR mesoscale environment at {VT[it]:%d %b %Y %H} UTC (12 PM CDT) "
             "\u2014 the tornado hour", x=0.085, ha="left", fontsize=12.5, y=1.0)
fig.text(0.085, 0.962, "Open triangles mark the seven NWS-confirmed tornadoes of "
         "27 July 2026; all seven touched down between 11:54 am and 1:05 pm CDT. "
         "Panels d\u2013e clipped at zero.", fontsize=8.5, color=META_GREY, ha="left")
fig.savefig("hrrr_environment_maps.png", dpi=200, bbox_inches="tight")

![HRRR mesoscale environment at 17 UTC (12 PM CDT), the tornado hour. Open triangles are the seven confirmed tornadoes.](01_tornadic_event_27jul2026_outputs/hrrr_environment_maps.png)

*HRRR mesoscale environment at 17 UTC (12 PM CDT), the tornado hour. Open triangles are the seven confirmed tornadoes.*

Panel a shows MLCAPE above 4000 J kg⁻¹ across essentially the whole metro, with the
domain maximum at 4820 J kg⁻¹. Panel b explains where it came from: 2 m dewpoints near
80 °F, which is tropical air rather than continental summer air, and which the
observational record confirms — O'Hare's daily maximum dewpoint climbed from 48 °F on
23 July to 79 °F on 27 July.

Panel c is the anomaly. Deep-layer shear above 50 kt over the metro, peaking near 65 kt
in the corridor where the tornadoes occurred, is a cool-season value appearing in late
July. Panel d puts the 0–1 km helicity maximum over DuPage and Cook counties, and panel
e shows the composite tornado parameter exceeding 3 along the damage corridor. Panel f
is the model's own reflectivity, which reproduces the discrete-cell mode: the HRRR was
not simulating a squall line at this hour, it was simulating separated supercells, and
that is what the radar shows.

The correspondence between the triangles and panels d–e is good but should not be
over-read. A 3 km model does not resolve tornadoes, and updraft helicity is a proxy for
a mesocyclone, not for a tornado. What panels d–e establish is that the model put the
favourable environment in the right place at the right hour, which is why we can use it
to reason about the forcing.

## 7. Time evolution: the environment kept improving after the tornadoes ended

The maps are a single hour. The interesting result is in the time series, so this
section tracks the environment at the Glen Ellyn / Wheaton corridor through the whole
event, alongside the O'Hare surface observations as a reality check on the model.

In [ ]:
def asos(stn, start, end):
    """IEM ASOS/METAR observations for one station."""
    u = ("https://mesonet.agron.iastate.edu/cgi-bin/request/asos.py?"
         f"station={stn}&data=tmpf&data=dwpf&data=sknt&data=gust&data=mslp&data=p01i&data=drct"
         f"&year1={start:%Y}&month1={start:%m}&day1={start:%d}&hour1={start:%H}&minute1=0"
         f"&year2={end:%Y}&month2={end:%m}&day2={end:%d}&hour2={end:%H}&minute2=0"
         "&tz=UTC&format=onlycomma&latlon=no&missing=M&trace=0.0001&direct=no"
         "&report_type=3&report_type=4")
    d = pd.read_csv(io.StringIO(requests.get(u, timeout=120).text), na_values=["M", "T"])
    d["valid"] = pd.to_datetime(d.valid, utc=True)
    return d


S, E = pd.Timestamp("2026-07-27 12:00"), pd.Timestamp("2026-07-28 06:00")
OBS = {}
for st in ["ORD", "MDW", "DPA", "LOT", "GYY", "PWK", "ARR"]:
    for attempt in range(3):        # the IEM CGI occasionally returns an error page
        try:
            d = asos(st, S, E)
            if "valid" not in d.columns:
                raise ValueError(f"unexpected columns: {list(d.columns)[:4]}")
            OBS[st] = d
            print(f"{st}: {len(d):3d} obs  maxgust {d.gust.max():4.0f} kt  "
                  f"maxTd {d.dwpf.max():4.0f}F  rain {d.p01i.sum():5.2f} in")
            break
        except Exception as ex:
            if attempt == 2:
                print(st, "FAIL", type(ex).__name__, str(ex)[:90])
            time.sleep(4)

In [ ]:
tt = [v - CDT for v in VT]              # model valid times in local (CDT)
o = OBS["ORD"]
ot = o.valid.dt.tz_localize(None) - CDT

plt.close("all")
apply_figure_style(sizes=(10, 9, 8))
fig, axs = plt.subplots(5, 1, figsize=(11.5, 13.6), sharex=True,
                        gridspec_kw=dict(hspace=0.30))


def shade(ax):
    """Pink: the tornado window. Blue: the second, linear, wind-damage round."""
    ax.axvspan(TORN_T0, TORN_T1, color="#b03060", alpha=0.13, lw=0, zorder=0)
    ax.axvspan(pd.Timestamp("2026-07-27 15:00"), pd.Timestamp("2026-07-27 17:00"),
               color="#4a6fa5", alpha=0.10, lw=0, zorder=0)


# (a) instability and inhibition
ax = axs[0]; shade(ax)
ax.plot(tt, at("cape_ml", *GL), color="#c0392b", marker="o", ms=3, label="MLCAPE")
ax.plot(tt, at("cape_sfc", *GL), color="#e08a5a", lw=1.0, ls="--", label="SBCAPE")
ax.set_ylabel("CAPE (J kg$^{-1}$)"); ax.set_ylim(0, 6100)
ax2 = ax.twinx()
ax2.plot(tt, at("cin_ml", *GL), color="#4a6fa5", lw=1.2, label="MLCIN")
ax2.set_ylabel("CIN (J kg$^{-1}$)", color="#4a6fa5")
ax2.tick_params(colors="#4a6fa5", labelsize=8)
ax2.set_ylim(-330, 15); ax2.spines["right"].set_visible(True)
ax.set_title("Instability builds through late morning; inhibition vanishes by 11 am",
             loc="left", pad=6)
h1, l1 = ax.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
ax.legend(h1 + h2, l1 + l2, loc="upper right", ncol=3, fontsize=8)

# (b) shear and helicity
ax = axs[1]; shade(ax)
ax.plot(tt, at("shr06_kt", *GL), color="#2471a3", marker="o", ms=3,
        label="0\u20136 km bulk shear")
ax.plot(tt, at("shr01_kt", *GL), color="#7fb3d5", lw=1.2, label="0\u20131 km bulk shear")
ax.set_ylabel("Bulk shear (kt)"); ax.set_ylim(0, 88)
ax3 = ax.twinx()
ax3.plot(tt, at("srh_01km", *GL), color="#7d3c98", lw=1.4, label="0\u20131 km SRH")
ax3.plot(tt, at("srh_03km", *GL), color="#c39bd3", lw=1.1, ls="--", label="0\u20133 km SRH")
ax3.set_ylabel("SRH (m$^2$ s$^{-2}$)", color="#7d3c98")
ax3.tick_params(colors="#7d3c98", labelsize=8)
ax3.spines["right"].set_visible(True); ax3.set_ylim(0, 980)
ax.set_title("Deep-layer shear near 65 kt, with a low-level helicity surge at the "
             "tornado hour", loc="left", pad=6)
h1, l1 = ax.get_legend_handles_labels(); h3, l3 = ax3.get_legend_handles_labels()
ax.legend(h1 + h3, l1 + l3, loc="upper right", ncol=2, fontsize=8)

# (c) composite indices
ax = axs[2]; shade(ax)
ax.plot(tt, at("stp", *GL), color="#b03060", marker="o", ms=3,
        label="Significant tornado parameter")
ax.plot(tt, at("scp", *GL) / 10.0, color="#1a7a3d", lw=1.3,
        label="Supercell composite \u00f7 10")
ax.axhline(1, color=META_GREY, lw=0.7, ls=":")
ax.text(tt[-1], 1.25, "STP = 1", fontsize=7, color=META_GREY, ha="right")
ax.set_ylabel("Composite index"); ax.set_ylim(0, 9.4)
ax.set_title("Tornado and supercell parameters keep climbing after the tornadoes end",
             loc="left", pad=6)
ax.legend(loc="upper right", ncol=2, fontsize=8)

# (d) O'Hare thermodynamics, model against observations
ax = axs[3]; shade(ax)
ax.plot(tt, at("t2f", *ORD), color="#c0392b", marker="o", ms=3, label="HRRR 2 m temperature")
ax.plot(tt, at("d2f", *ORD), color="#1a7a3d", marker="o", ms=3, label="HRRR 2 m dewpoint")
ax.plot(ot, o.tmpf, color="#c0392b", ls="none", marker="x", ms=4.5, alpha=0.8,
        label="O'Hare observed T")
ax.plot(ot, o.dwpf, color="#1a7a3d", ls="none", marker="x", ms=4.5, alpha=0.8,
        label="O'Hare observed Td")
ax.set_ylabel("Temperature (\u00b0F)"); ax.set_ylim(56, 100)
ax4 = ax.twinx()
ax4.plot(tt, at("pwat_in", *ORD), color="#8e6e3c", lw=1.4, label="Precipitable water")
ax4.set_ylabel("PWAT (in)", color="#8e6e3c")
ax4.tick_params(colors="#8e6e3c", labelsize=8)
ax4.spines["right"].set_visible(True); ax4.set_ylim(1.2, 2.75)
ax.set_title("O'Hare: HRRR tracks observed temperature and near-80 \u00b0F dewpoints",
             loc="left", pad=6)
h1, l1 = ax.get_legend_handles_labels(); h4, l4 = ax4.get_legend_handles_labels()
ax.legend(h1 + h4, l1 + l4, loc="lower left", ncol=3, fontsize=7.5)

# (e) rain and gusts
ax = axs[4]; shade(ax)
ax.bar(tt, at("apcp_in", *ORD), width=0.028, color="#3f7fbf",
       label="HRRR hourly precipitation", zorder=3)
ax.set_ylabel("Hourly precip (in)"); ax.set_ylim(0, 1.05)
ax5 = ax.twinx()
ax5.plot(tt, at("gust_mph", *ORD), color="#d35400", lw=1.4, label="HRRR wind gust")
gobs = o.dropna(subset=["gust"])
ax5.plot(gobs.valid.dt.tz_localize(None) - CDT, gobs.gust * 1.15078, ls="none",
         marker="x", ms=5, color="#7e2f00", label="O'Hare observed gust")
ax5.set_ylabel("Gust (mph)", color="#d35400")
ax5.tick_params(colors="#d35400", labelsize=8)
ax5.spines["right"].set_visible(True); ax5.set_ylim(0, 105)
ax.set_title("Heavy rain rates and damaging gusts accompany both storm rounds",
             loc="left", pad=6)
h1, l1 = ax.get_legend_handles_labels(); h5, l5 = ax5.get_legend_handles_labels()
ax.legend(h1 + h5, l1 + l5, loc="upper right", ncol=3, fontsize=8)

axs[-1].xaxis.set_major_formatter(mdates.DateFormatter("%-I %p\n%a"))
axs[-1].xaxis.set_major_locator(mdates.HourLocator(interval=3))
axs[-1].set_xlabel("Local time (CDT), 27\u201328 July 2026")
axs[-1].set_xlim(tt[0], tt[-1])
for k, ax in enumerate(axs):
    panel_letter(ax, "abcde"[k], dx=-0.075, dy=1.03)

fig.suptitle("HRRR meteogram \u2014 Glen Ellyn / Wheaton tornado corridor (a\u2013c) "
             "and Chicago O'Hare (d\u2013e)", x=0.075, ha="left", fontsize=12.5, y=0.951)
fig.text(0.075, 0.918, "Pink band: tornado window 11:54 am\u20131:05 pm CDT.   "
         "Blue band: second wind-damage round, 3\u20135 pm CDT.",
         fontsize=8.5, color=META_GREY, ha="left")
fig.savefig("hrrr_meteogram.png", dpi=200, bbox_inches="tight")

![HRRR meteogram at the Glen Ellyn / Wheaton corridor (a-c) and O'Hare (d-e). Pink band is the tornado window; blue band the second, linear round.](01_tornadic_event_27jul2026_outputs/hrrr_meteogram.png)

*HRRR meteogram at the Glen Ellyn / Wheaton corridor (a-c) and O'Hare (d-e). Pink band is the tornado window; blue band the second, linear round.*

In [ ]:
# The central timing result, stated numerically.
stp_gl = at("stp", *GL)
i16 = list(VT).index(pd.Timestamp("2026-07-27 17:00"))   # 12 pm CDT, tornado hour
i21 = list(VT).index(pd.Timestamp("2026-07-27 21:00"))   #  4 pm CDT, second round
print(f"STP at the tornado hour (12 pm CDT): {stp_gl[i16]:.1f}")
print(f"STP at 4 pm CDT (second round)     : {stp_gl[i21]:.1f}")
print(f"STP maximum over the whole event   : {stp_gl.max():.1f} "
      f"at {(VT[int(np.argmax(stp_gl))] - CDT):%-I %p} CDT")

This is the result that most complicates the simple narrative. The significant tornado
parameter at the Glen Ellyn corridor was **4.2 at the tornado hour** and rose to
**7.7 by 4 pm CDT** — its maximum for the event, three hours *after* the last tornado
lifted. By the conventional reading of composite indices, the most tornado-favourable
environment of the day went unused.

What changed was storm mode. The midday round was a family of discrete supercells; by
mid-afternoon convection had congealed into a bowing line, which produces damaging
straight-line wind rather than tornadoes. That is why the great majority of the day's
damage — including the **82 kt (94 mph)** measured gust at Beach Park — was
straight-line wind despite seven confirmed tornadoes, and it is the honest caveat on
any operational use of composite parameters. STP tells you whether the environment
*could* support a significant tornado. It cannot tell you whether the convection will
be organised in a way that uses it. On this day the two were out of phase.

Panel d is also a model verification: the HRRR 2 m temperature and dewpoint track the
O'Hare observations closely through the whole period, which is the licence for trusting
the model soundings in the next section.

## 8. Model soundings through the event, and a trap in the 4 pm profile

Four HRRR soundings at the Glen Ellyn / Wheaton corridor: pre-convective, destabilising,
the tornado hour, and the second round. Model soundings from pressure-level output have
a known weakness near the ground — the lowest isobaric level can be below the surface,
and the true 2 m state is a separate field — so the profile-building function splices
the actual 2 m temperature, 2 m dewpoint and 10 m wind onto the isobaric column and
discards levels at or beneath the surface pressure.

In [ ]:
PR = pickle.load(open("hrrr_prs.pkl", "rb"))
PLAT, PLON, LEV = PR["lat"], PR["lon"], PR["levels"]


def pcell(lat, lon):
    return np.unravel_index(np.argmin((PLAT - lat) ** 2 + (PLON - lon) ** 2), PLAT.shape)


def hrrr_sounding(lat, lon, tkey, tidx):
    """Isobaric profile with the true 2 m / 10 m surface spliced on.

    tkey indexes the pressure-level pickle; tidx indexes the hourly surface arrays.
    """
    d = PR[tkey]
    j, i = pcell(lat, lon)
    T = d["t"][:, j, i] - 273.15
    Td = d["dpt"][:, j, i] - 273.15
    zz = d["gh"][:, j, i]
    uu = d["u"][:, j, i] * 1.94384
    vv = d["v"][:, j, i] * 1.94384

    psfc = float(at("mslp_hpa", lat, lon, tidx))
    t2 = float(at("t2m", lat, lon, tidx)) - 273.15
    d2 = float(at("d2m", lat, lon, tidx)) - 273.15
    u10 = float(at("u10", lat, lon, tidx)) * 1.94384
    v10 = float(at("v10", lat, lon, tidx)) * 1.94384

    keep = LEV < psfc - 2                      # drop sub-surface levels
    P_ = np.concatenate([[psfc], LEV[keep]])
    TT = np.concatenate([[t2], T[keep]])
    DD = np.concatenate([[d2], Td[keep]])
    ZZ = np.concatenate([[float(np.interp(-psfc, -LEV, zz))], zz[keep]])
    UU = np.concatenate([[u10], uu[keep]])
    VV = np.concatenate([[v10], vv[keep]])
    s = np.argsort(-P_)
    return (P_[s] * units.hPa, TT[s] * units.degC, DD[s] * units.degC,
            ZZ[s] * units.m, UU[s] * units.knots, VV[s] * units.knots)


def hrrr_params(p, T, Td, z, u, v):
    """Derived parameters for a model sounding. No try/except here: unlike radiosondes,
    the model column is complete, so a failure is a bug and should surface."""
    o = {}
    for lab, fn in [("SB", mpcalc.surface_based_cape_cin),
                    ("MU", mpcalc.most_unstable_cape_cin)]:
        c, ci = fn(p, T, Td)
        o[lab + "CAPE"], o[lab + "CIN"] = c.m, ci.m
    ml, mlc = mpcalc.mixed_layer_cape_cin(p, T, Td, depth=100 * units.hPa)
    o["MLCAPE"], o["MLCIN"] = ml.m, mlc.m
    lp, _ = mpcalc.lcl(p[0], T[0], Td[0])
    o["LCL_hPa"] = lp.m
    o["LCL_m"] = float(np.interp(-lp.m, -p.m, z.m) - z[0].m)
    o["PWAT_mm"] = mpcalc.precipitable_water(p, Td).to("mm").m
    for top, lab in [(1000, "01"), (3000, "03"), (6000, "06")]:
        bs = mpcalc.bulk_shear(p, u, v, height=z, depth=top * units.m)
        o[f"SHR{lab}_kt"] = float(mpcalc.wind_speed(*bs).to("knots").m)
    rm, lm, mn = mpcalc.bunkers_storm_motion(p, u, v, z)
    o["BunkR_kt"] = float(mpcalc.wind_speed(*rm).to("knots").m)
    o["BunkR_dir"] = float(mpcalc.wind_direction(*rm).m)
    for top, lab in [(1000, "01"), (3000, "03")]:
        s = mpcalc.storm_relative_helicity(z, u, v, depth=top * units.m,
                                           storm_u=rm[0], storm_v=rm[1])
        o[f"SRH{lab}"] = float(s[2].m)
    zz = z.m - z[0].m
    o["LR_0_3km"] = float(-(np.interp(3000, zz, T.m) - T[0].m) / 3.0)
    t7, t5 = np.interp(-700, -p.m, T.m), np.interp(-500, -p.m, T.m)
    z7, z5 = np.interp(-700, -p.m, z.m), np.interp(-500, -p.m, z.m)
    o["LR_700_500"] = float(-(t5 - t7) / ((z5 - z7) / 1000.0))
    return o, rm


SND_TIMES = [("2026-07-27 12:00", 0, "7 am CDT \u2014 pre-convective"),
             ("2026-07-27 15:00", 3, "10 am CDT \u2014 destabilising"),
             ("2026-07-27 17:00", 5, "12 pm CDT \u2014 tornado hour"),
             ("2026-07-27 21:00", 9, "4 pm CDT \u2014 second round")]

hres = {}
for tk, ti, lab in SND_TIMES:
    prof = hrrr_sounding(*GL, str(pd.Timestamp(tk)), ti)
    hres[lab], _ = hrrr_params(*prof)
print(pd.DataFrame(hres).T[["MUCAPE", "MLCAPE", "MLCIN", "LCL_m", "SHR06_kt", "SRH01"]]
      .round(0).to_string())

In [ ]:
OUTFLOW_NOTE = (
    "Panel d samples rain-cooled outflow: surface $\\theta_e$ falls 367\u2192345 K and 2 m "
    "temperature drops 7.5 \u00b0C between 2 and 4 pm \u2014\n"
    "during peak solar heating \u2014 after 0.31 in of rain. Its low MLCAPE describes the "
    "outflow, not the inflow feeding the storms.")

plt.close("all")
apply_figure_style(sizes=(9, 8, 7))
figs = plt.figure(figsize=(13.5, 13.2))
gsx = gridspec.GridSpec(2, 2, figure=figs, hspace=0.36, wspace=0.20, top=0.875)

for k, (tk, ti, lab) in enumerate(SND_TIMES):
    P_, T_, Td_, Z_, U_, V_ = hrrr_sounding(*GL, str(pd.Timestamp(tk)), ti)
    m = P_.m >= 120
    o = hres[lab]
    stats = (f"MUCAPE {o['MUCAPE']:.0f} J kg$^{{-1}}$\nMLCAPE {o['MLCAPE']:.0f} J kg$^{{-1}}$\n"
             f"MLCIN {o['MLCIN']:.0f} J kg$^{{-1}}$\nLCL {o['LCL_m']:.0f} m\n"
             f"PWAT {o['PWAT_mm']:.0f} mm\n0\u20136 km shear {o['SHR06_kt']:.0f} kt\n"
             f"0\u20131 km SRH {o['SRH01']:.0f} m$^2$ s$^{{-2}}$")
    skew = skew_panel(figs, gsx[k], P_, T_, Td_, U_[m][::2], V_[m][::2], P_[m][::2],
                      lab, "HRRR at Glen Ellyn / Wheaton", stats, "abcd"[k])
    hm = (Z_.m - Z_.m[0]) <= 9000
    hodo_inset(skew.ax, U_[hm], V_[hm], (Z_.m[hm] - Z_.m[0]) * units.m,
               component_range=70)

figs.legend(handles=SND_LEGEND, loc="lower center", ncol=5, frameon=False,
            bbox_to_anchor=(0.5, 0.048), fontsize=8.5)
figs.suptitle("HRRR model soundings through the event \u2014 Glen Ellyn / Wheaton "
              "tornado corridor", x=0.09, ha="left", fontsize=12, y=0.978)
figs.text(0.09, 0.955, OUTFLOW_NOTE, fontsize=8.2, color=META_GREY, ha="left", va="top")
figs.savefig("hrrr_soundings.png", dpi=200, bbox_inches="tight")

![HRRR model soundings at the Glen Ellyn / Wheaton corridor: (a) 7 am, (b) 10 am, (c) 12 pm tornado hour, (d) 4 pm second round. Panel d samples rain-cooled outflow.](01_tornadic_event_27jul2026_outputs/hrrr_soundings.png)

*HRRR model soundings at the Glen Ellyn / Wheaton corridor: (a) 7 am, (b) 10 am, (c) 12 pm tornado hour, (d) 4 pm second round. Panel d samples rain-cooled outflow.*

Panels a through c are the destabilisation sequence: a capped, saturated boundary layer
at 7 am; the cap eroding by 10 am; and by noon a deep, nearly-uncapped profile with a
390 m LCL and a strongly curved low-level hodograph. A low LCL matters for tornadoes
specifically — it keeps the downdraft air from becoming too negatively buoyant, which is
one of the more robust discriminators between tornadic and non-tornadic supercells.

**Panel d is a trap, and it is worth being explicit about it.** Read naively, its
MLCAPE of 420 J kg⁻¹ says the atmosphere had stabilised by 4 pm and the second round of
storms had little to work with. That reading is wrong. The 4 pm column is not storm
inflow — it is rain-cooled outflow that the earlier convection had already laid down
over this grid point. Between 2 pm and 4 pm, during *peak solar heating*, surface
equivalent potential temperature at this point collapses from 367 K to 345 K and the
2 m temperature falls 7.5 °C, after 0.31 in of rain. Air does not do that under a
July sun unless a downdraft has replaced it.

So the 420 J kg⁻¹ describes the cold pool, not the inflow. It is exactly the situation
where a single-point model sounding will mislead you: the point is inside the storm's own
wake. The correct storm-relevant environment for the 4 pm round is the one immediately
ahead of the outflow boundary, which is what the mapped STP field of 7.7 was showing.
The two numbers are not in conflict; they are sampling different air.

## 9. Forcing: what actually made the storms go up

Instability and shear are the *capacity* for severe convection. They do not by themselves
initiate anything: a capped, strongly-sheared, high-CAPE airmass can sit there all day.
Something has to lift a parcel to its level of free convection. This section identifies
the trigger.

The answer has two parts, one synoptic and one mesoscale. Aloft, a weak 500 hPa
vorticity lobe drifted over northern Illinois — the "subtle disturbance" of the forecast
discussion, not a closed low or a strong trough. At the surface, a sharp equivalent
potential temperature gradient marked a boundary that behaved as a lake-breeze front.
The mid-level lobe supplied gentle, broad ascent; the surface boundary supplied the
focused low-level convergence. All seven touchdowns fall along that boundary.

In [ ]:
def lev_index(x):
    return int(np.argmin(np.abs(LEV - x)))


i500, i700, i850, i925 = lev_index(500), lev_index(700), lev_index(850), lev_index(925)
TK = str(pd.Timestamp("2026-07-27 17:00"))
dP = PR[TK]

# 850 hPa equivalent potential temperature: the moist conveyor.
the850 = mpcalc.equivalent_potential_temperature(
    LEV[i850] * units.hPa,
    (dP["t"][i850] - 273.15) * units.degC,
    (dP["dpt"][i850] - 273.15) * units.degC).m

# Surface theta-e: resolves the lake-breeze / outflow boundary.
the_sfc = mpcalc.equivalent_potential_temperature(
    (M["mslp"][it] / 100.0) * units.hPa,
    (M["t2m"][it] - 273.15) * units.degC,
    (M["d2m"][it] - 273.15) * units.degC).m

# 700 hPa vertical velocity MUST come from F01, not the F00 analysis in hrrr_prs.pkl:
# the analysis has no spun-up convection and its ascent is ~38x too weak.
if "w_f01" in dP:
    w_f01 = dP["w_f01"]
else:
    from herbie import Herbie
    j0, j1, i0, i1 = PR["j0"], PR["j1"], PR["i0"], PR["i1"]
    Hw = Herbie(pd.Timestamp("2026-07-27 16:00"), model="hrrr", product="prs", fxx=1,
                save_dir=SAVE, verbose=False)
    rw = Hw.xarray(r":VVEL:\d+ mb:", remove_grib=False)
    dsw = rw[0] if isinstance(rw, list) else rw
    w_f01 = dsw["w"].values[:, j0:j1, i0:i1].astype("float32")

print(f"700 hPa peak ascent, F01: {np.nanmin(w_f01[i700]):.1f} Pa/s   "
      f"F00: {np.nanmin(dP['w'][i700]):.1f} Pa/s   "
      f"ratio {np.nanmin(w_f01[i700]) / np.nanmin(dP['w'][i700]):.0f}x")

In [ ]:
def basemap_syn(ax):
    ax.set_extent([DOM_SYN["lon_min"], DOM_SYN["lon_max"],
                   DOM_SYN["lat_min"], DOM_SYN["lat_max"]], crs=PC)
    ax.add_feature(cfeature.STATES.with_scale("50m"), edgecolor="0.25", lw=0.7, zorder=6)
    ax.add_feature(cfeature.LAKES.with_scale("50m"), fc="none", ec="0.1", lw=0.9, zorder=6)


plt.close("all")
apply_figure_style(sizes=(9.5, 8.5, 7.5))
F = plt.figure(figsize=(14.5, 9.2))
G = gridspec.GridSpec(2, 2, figure=F, hspace=0.20, wspace=0.10)
specs = []

# (a) 500 hPa absolute vorticity and heights -- the synoptic trigger
ax = F.add_subplot(G[0], projection=PC); basemap_syn(ax)
cf = ax.contourf(PLON, PLAT, gaussian_filter(dP["absv"][i500] * 1e5, 8.0),
                 levels=np.arange(4, 22, 0.75), cmap="ChaseSpectral", extend="both",
                 transform=PC, zorder=3)
cs = ax.contour(PLON, PLAT, gaussian_filter(dP["gh"][i500], 6),
                levels=np.arange(5760, 6000, 30), colors="k", linewidths=0.9,
                transform=PC, zorder=5)
ax.clabel(cs, fmt="%d", fontsize=6.5, inline=True)
specs.append((ax, cf, "500 hPa absolute vorticity ($10^{-5}$ s$^{-1}$), heights (m)",
              "Weak mid-level vorticity lobe \u2014 the 'subtle disturbance'", "a"))

# (b) 700 hPa vertical velocity from F01 -- where convection is actually going up
ax = F.add_subplot(G[1], projection=PC); basemap_syn(ax)
cf = ax.contourf(PLON, PLAT, gaussian_filter(w_f01[i700], 1.0),
                 levels=np.arange(-12, 12.1, 1), cmap="balance_r", extend="both",
                 transform=PC, zorder=3)
specs.append((ax, cf, "700 hPa vertical velocity (Pa s$^{-1}$; blue = ascent)",
              "Deep convective ascent focused on a narrow corridor", "b"))

# (c) 850 hPa theta-e and winds -- the moist conveyor
ax = F.add_subplot(G[2], projection=PC); basemap_syn(ax)
cf = ax.contourf(PLON, PLAT, gaussian_filter(the850, 1.5), levels=np.arange(320, 368, 2),
                 cmap="HomeyerRainbow", extend="both", transform=PC, zorder=3)
sk = 9
ax.barbs(PLON[::sk, ::sk], PLAT[::sk, ::sk],
         dP["u"][i850][::sk, ::sk] * 1.94384, dP["v"][i850][::sk, ::sk] * 1.94384,
         length=4.6, lw=0.5, transform=PC, zorder=7)
specs.append((ax, cf, "850 hPa equivalent potential temperature (K), winds (kt)",
              "Warm moist conveyor: high-$\\theta_e$ air fed north into the metro", "c"))

# (d) surface theta-e, MSLP and 10 m winds -- the mesoscale trigger
ax = F.add_subplot(G[3], projection=PC)
basemap(ax, label_cities=False)
cf = ax.contourf(LON, LAT, gaussian_filter(the_sfc, 1.0), levels=np.arange(330, 375, 1.5),
                 cmap="HomeyerRainbow", extend="both", transform=PC, zorder=3)
cs = ax.contour(LON, LAT, gaussian_filter(M["mslp_hpa"][it], 2),
                levels=np.arange(996, 1016, 1), colors="k", linewidths=0.8,
                transform=PC, zorder=5)
ax.clabel(cs, fmt="%d", fontsize=6.5, inline=True)
sk = 6
ax.barbs(LON[::sk, ::sk], LAT[::sk, ::sk],
         M["u10"][it][::sk, ::sk] * 1.94384, M["v10"][it][::sk, ::sk] * 1.94384,
         length=4.4, lw=0.5, transform=PC, zorder=7)
torn_marks(ax, ms=70, lw=1.4)
specs.append((ax, cf, "Surface $\\theta_e$ (K), MSLP (hPa), 10 m winds (kt)",
              "Tornadoes track along the surface $\\theta_e$ / lake-breeze boundary", "d"))

for ax, cf, cblabel, title, L in specs:
    cb = F.colorbar(cf, ax=ax, orientation="horizontal", pad=0.03, aspect=32, shrink=0.92)
    cb.set_label(cblabel, fontsize=8)
    cb.ax.tick_params(labelsize=7)
    ax.set_title(title, loc="left", pad=13)
    panel_letter(ax, L, dx=-0.012, dy=1.10)

F.suptitle("Forcing mechanisms at 27 Jul 2026 17 UTC (12 PM CDT)", x=0.08, ha="left",
           fontsize=12.5, y=1.0)
F.text(0.08, 0.955, "Panels a, c, d: HRRR analysis. Panel b: 1-h forecast \u2014 the "
       "analysis carries no spun-up convection. Triangles in d mark the seven confirmed "
       "tornadoes.", fontsize=8.5, color=META_GREY, ha="left")
F.savefig("hrrr_synoptic_forcing.png", dpi=200, bbox_inches="tight")

![Forcing mechanisms at 17 UTC. (a) 500 hPa absolute vorticity and heights, (b) 700 hPa vertical velocity from the F01 forecast, (c) 850 hPa theta-e and winds, (d) surface theta-e, MSLP and 10 m winds with the seven tornado locations.](01_tornadic_event_27jul2026_outputs/hrrr_synoptic_forcing.png)

*Forcing mechanisms at 17 UTC. (a) 500 hPa absolute vorticity and heights, (b) 700 hPa vertical velocity from the F01 forecast, (c) 850 hPa theta-e and winds, (d) surface theta-e, MSLP and 10 m winds with the seven tornado locations.*

Panel a shows how unremarkable the synoptic pattern was: a weak absolute vorticity lobe
over northern Illinois with nearly flat 500 hPa heights. There is no deep trough, no
strong jet streak, nothing that would headline a severe-weather outlook. This is the
"subtle disturbance" case, and it is precisely the kind of setup that is hardest to
forecast — the trigger is weak, so whether anything happens depends on mesoscale detail.

Panel b, from the F01 forecast rather than the analysis, shows where that detail put the
convection: a narrow corridor of strong 700 hPa ascent. The contrast with the F00
analysis is the reason the fetch uses F01 at all; the analysis ascent is weaker by more
than an order of magnitude, because the analysis simply does not contain the storms.

Panel c is the moisture supply, high-θₑ air advected north into the metro on a
low-level jet. Panel d is the trigger, and the key panel of the figure: a sharp surface
θₑ gradient running northwest to southeast across the metro, with the 10 m wind field
converging across it, and **all seven tornado locations lying along it**. Lake Michigan
sits immediately east, and the boundary has the signature of a lake-breeze front —
the cool, stable lake air undercutting the tropical inland airmass and providing a
persistent line of low-level convergence right where the shear and helicity were
maximised.

That coincidence is the mechanism. A weak upper disturbance provided broad lift, the
lake breeze provided the focus, and the airmass supplied the rest. Change the position
of that boundary by 30 km and the tornado corridor moves with it.

## 10. What was observed: storm reports

The model tells a coherent story. The next two sections check it against what people on
the ground actually reported and what the damage surveys actually found.

In [ ]:
lsr_url = ("https://mesonet.agron.iastate.edu/cgi-bin/request/gis/lsr.py?"
           "sts=2026-07-27T12:00Z&ets=2026-07-28T12:00Z"
           "&wfo[]=LOT&wfo[]=MKX&wfo[]=IWX&wfo[]=DVN&wfo[]=ILX&fmt=csv")
lsr = pd.read_csv(io.StringIO(requests.get(lsr_url, timeout=120).text))
lsr["VALID"] = pd.to_datetime(lsr["VALID"], format="%Y%m%d%H%M", utc=True)
lsr["cdt"] = lsr["VALID"].dt.tz_convert("America/Chicago")
lsr = lsr.sort_values("VALID")
lsr.to_csv("lsr_20260727.csv", index=False)

print(f"{len(lsr)} local storm reports")
print(lsr.TYPETEXT.value_counts().head(8).to_string())
print(f"max hail {lsr.loc[lsr.TYPETEXT == 'HAIL', 'MAG'].max():.2f} in   "
      f"max gust {lsr.loc[lsr.TYPETEXT == 'TSTM WND GST', 'MAG'].max():.0f} kt")

In [ ]:
lsr_wind = lsr[lsr.TYPETEXT.isin(["TSTM WND DMG", "TSTM WND GST"])]
lsr_hail = lsr[lsr.TYPETEXT == "HAIL"]

plt.close("all")
apply_figure_style(sizes=(10, 9, 8))
fig = plt.figure(figsize=(14.5, 6.6))
gs = gridspec.GridSpec(1, 2, figure=fig, width_ratios=[1.15, 1.0], wspace=0.16)

# (a) reports over the modelled rotation swath
ax = fig.add_subplot(gs[0], projection=PC)
basemap(ax, label_cities=True)
uh_swath = M["uh_25km"][:11].max(axis=0)          # 12-22 UTC, hourly maxima
cf = ax.contourf(LON, LAT, np.where(uh_swath < 25, np.nan, uh_swath),
                 levels=np.arange(25, 401, 25), cmap="ChaseSpectral", alpha=0.85,
                 transform=PC, zorder=3, extend="max")
cb = fig.colorbar(cf, ax=ax, orientation="horizontal", pad=0.035, aspect=34, shrink=0.9)
cb.set_label("HRRR maximum 2\u20135 km updraft helicity, 12\u201322 UTC (m$^2$ s$^{-2}$)",
             fontsize=8)
cb.ax.tick_params(labelsize=7)

in_dom = lambda d: d[d.LAT.between(DOM["lat_min"], DOM["lat_max"])
                     & d.LON.between(DOM["lon_min"], DOM["lon_max"])]
w, hl = in_dom(lsr_wind), in_dom(lsr_hail)
ax.scatter(w.LON, w.LAT, s=13, c="#1f4e8c", marker="s", alpha=0.75, lw=0, transform=PC,
           zorder=7, label=f"Wind damage / gust ({len(w)})")
ax.scatter(hl.LON, hl.LAT, s=16, c="#00a86b", marker="^", alpha=0.8, lw=0, transform=PC,
           zorder=7, label=f"Hail ({len(hl)})")
torn_marks(ax, ms=95, lw=1.6)
ax.scatter([], [], marker="v", s=95, c="none", edgecolors="k", lw=1.6,
           label="Confirmed tornado (7)")
ax.legend(loc="lower left", fontsize=7.8, framealpha=0.9, frameon=True,
          facecolor="white", edgecolor="0.8")
ax.set_title("Damage reports fall along the modelled rotation swath", loc="left", pad=8)
panel_letter(ax, "a", dx=-0.01, dy=1.06, fontsize=12)

# (b) hazard timeline
axb = fig.add_subplot(gs[1])
allr = lsr[lsr.LAT.between(39.8, 43.8) & lsr.LON.between(-92, -85)].copy()
allr["hr"] = allr.cdt.dt.floor("1h")
cats = [("TORNADO", "Tornado", "#b03060"), ("HAIL", "Hail", "#00a86b"),
        ("TSTM WND DMG", "Wind damage", "#1f4e8c"), ("TSTM WND GST", "Measured gust", "#7fb3d5"),
        ("FLASH FLOOD", "Flash flood", "#8e6e3c"), ("FLOOD", "Flood", "#c9a227")]
hrs = pd.date_range(allr.hr.min(), allr.hr.max(), freq="1h")
bot = np.zeros(len(hrs))
for code_, lab, col in cats:
    cnt = allr[allr.TYPETEXT == code_].groupby("hr").size().reindex(hrs, fill_value=0).values
    axb.bar([h.tz_localize(None) for h in hrs], cnt, bottom=bot, width=0.034,
            color=col, label=f"{lab} ({cnt.sum()})", zorder=3)
    bot += cnt
axb.set_ylabel("NWS storm reports per hour")
axb.set_xlabel("Local time (CDT), 27 July 2026")
axb.xaxis.set_major_formatter(mdates.DateFormatter("%-I %p"))
axb.xaxis.set_major_locator(mdates.HourLocator(interval=2))
axb.set_ylim(0, 132)
axb.legend(loc="upper right", fontsize=8, ncol=2, bbox_to_anchor=(1.0, 1.005))
axb.axvspan(TORN_T0, TORN_T1, color="#b03060", alpha=0.12, lw=0, zorder=0)
axb.annotate("All 7 tornadoes\n11:54 am \u2013 1:05 pm",
             xy=(mdates.date2num(pd.Timestamp("2026-07-27 12:15")), 106), xycoords="data",
             xytext=(0.10, 0.60), textcoords="axes fraction", fontsize=8,
             arrowprops=dict(arrowstyle="->", lw=0.9, color="#b03060",
                             connectionstyle="arc3,rad=-0.2"),
             color="#b03060", va="center", ha="left")
axb.annotate("Second round:\nbowing line, 82 kt gust",
             xy=(mdates.date2num(pd.Timestamp("2026-07-27 15:35")), 60), xycoords="data",
             xytext=(0.66, 0.62), textcoords="axes fraction", fontsize=8,
             arrowprops=dict(arrowstyle="->", lw=0.9, color="#1f4e8c"), color="#1f4e8c")
axb.set_title("Two distinct rounds: midday supercells, then an afternoon wind line",
              loc="left", pad=8)
axb.margins(x=0.02)
panel_letter(axb, "b", dx=-0.09, dy=1.06, fontsize=12)

fig.suptitle("Observed hazard reports, 27 July 2026 \u2014 Chicagoland", x=0.075,
             ha="left", fontsize=12.5, y=1.0)
fig.text(0.075, 0.955, f"{len(allr)} NWS local storm reports in the domain; largest hail "
         "3.00 in, strongest measured gust 82 kt (94 mph) at Beach Park.",
         fontsize=8.5, color=META_GREY, ha="left")
fig.savefig("storm_reports.png", dpi=200, bbox_inches="tight")

![Observed hazard reports. (a) NWS local storm reports over the HRRR maximum 2-5 km updraft helicity swath, 12-22 UTC. (b) Reports per hour by hazard type.](01_tornadic_event_27jul2026_outputs/storm_reports.png)

*Observed hazard reports. (a) NWS local storm reports over the HRRR maximum 2-5 km updraft helicity swath, 12-22 UTC. (b) Reports per hour by hazard type.*

There were **311 NWS local storm reports** in the domain over the 24 hours beginning
12Z. Panel a shows their spatial distribution against the model's accumulated 2–5 km
updraft helicity swath, and the correspondence is good: the reports cluster along the
modelled rotation track, which is the strongest available evidence that the HRRR had the
mesoscale placement right and not merely the airmass.

Panel b is the timing, and it makes the two-round structure unambiguous. A first,
sharp spike of reports at midday contains every one of the seven tornadoes. Report
counts then *increase* into mid-afternoon, but the composition changes: the second peak
is wind damage and measured gusts, with the day's strongest measured gust of 82 kt
(94 mph) at Beach Park, and the largest hail of the day at 3.00 in near Shorewood. By
evening, flood and flash-flood reports begin to replace wind reports, which is where
the second notebook in this series picks up.

The caveat on storm reports is well known but worth restating: they are a
population-weighted sample of damage, not a measurement of the atmosphere. Reports
cluster where people and property are, and the Chicago metro guarantees dense reporting.
Absence of reports over rural Grundy or Kankakee counties is not evidence of absence of
hazard.

## 11. The surveyed damage tracks

Storm reports are points. The NWS damage surveys produce polylines with start and end
times, path length, maximum width and estimated maximum wind. These are the
authoritative record of what happened, and they are what the tornado climatology is
eventually built from.

> **Warning — the public DAT viewer is not the API.**
>
> The obvious entry point, `https://apps.dat.noaa.gov/`, is a JavaScript application.
> Its apparent REST paths return 404 when queried directly, and time spent trying to
> reverse-engineer them is wasted. The working service is a plain ArcGIS FeatureServer
> at
> `https://services.dat.noaa.gov/arcgis/rest/services/nws_damageassessmenttoolkit/DamageViewer/FeatureServer`.
>
> Use **layer 1, "Damage Lines"**. It carries full polyline geometry plus `efscale`,
> `starttime`/`endtime` (epoch milliseconds), `length`, `width` and `maxwind` — everything
> needed. Layer 0, the damage *points* layer, returned **zero rows** for this event, so
> do not treat an empty layer-0 response as "the survey is not published yet".

In [ ]:
DAT = ("https://services.dat.noaa.gov/arcgis/rest/services/"
       "nws_damageassessmenttoolkit/DamageViewer/FeatureServer/1/query")

t0 = pd.Timestamp("2026-07-27 00:00", tz="UTC")
t1 = pd.Timestamp("2026-07-28 12:00", tz="UTC")
params = {
    "where": "1=1",
    "geometry": "-88.8,41.1,-87.1,42.3",           # lon_min,lat_min,lon_max,lat_max
    "geometryType": "esriGeometryEnvelope",
    "inSR": 4326, "outSR": 4326,
    "spatialRel": "esriSpatialRelIntersects",
    "time": f"{int(t0.value // 10**6)},{int(t1.value // 10**6)}",
    "outFields": "*", "returnGeometry": "true", "f": "geojson",
}
gj = requests.get(DAT, params=params, timeout=180).json()
json.dump(gj, open("tornado_tracks_dat.geojson", "w"))

recs = []
for f in gj["features"]:
    p, g = f["properties"], f["geometry"]
    if g is None or g["type"] != "LineString":
        continue
    recs.append(dict(
        label=(p.get("event_name") or p.get("location") or "").strip(),
        ef=int(re.sub(r"\D", "", str(p["efscale"])) or 0),
        t0=pd.to_datetime(p["starttime"], unit="ms", utc=True),
        t1=pd.to_datetime(p["endtime"], unit="ms", utc=True),
        length_mi=float(p["length"]), width_yd=float(p["width"]),
        maxwind=float(p["maxwind"]),
        coords=g["coordinates"]))

TR = pd.DataFrame(recs).sort_values("t0").reset_index(drop=True)
TR.drop(columns="coords").to_csv("tornado_tracks_dat.csv", index=False)
print(f"{len(TR)} surveyed damage lines")
print(TR.assign(t0_cdt=TR.t0.dt.tz_convert("America/Chicago").dt.strftime("%-I:%M %p"))
      [["label", "ef", "t0_cdt", "length_mi", "width_yd", "maxwind"]].to_string(index=False))
print(f"\ntotal path length {TR.length_mi.sum():.1f} mi   widest {TR.width_yd.max():.0f} yd"
      f"   peak estimated wind {TR.maxwind.max():.0f} mph")

> **Warning — the Country Club Hills–Glenwood track really does loop. Do not "fix" it.**
>
> The EF1 that began at 12:47 pm CDT, locally nicknamed the "drunknado", traces a
> genuine counterclockwise loop roughly 2.8 km across in the middle of its track before
> resuming its southeastward motion. Plotted raw, the geometry looks like a
> vertex-ordering error, and the natural instinct is to reorder the vertices — by
> projecting them onto their principal axis, sorting along it, and reconnecting.
>
> **That instinct produces a wrong answer.** PCA reordering inflates this path to
> **46 km**. The as-stored vertex order gives **18.35 km**, which matches the survey's
> own recorded length of **18.41 km** to better than half a percent. The stored order is
> correct and the loop is real. The check below is the one to run before touching any
> survey geometry: compute the geodesic path length in the order the service gave you
> and compare it with the `length` attribute. If they agree, the vertex order is right,
> however strange the shape looks.

In [ ]:
def path_length_km(coords):
    """Geodesic length of a lon/lat polyline in the order given."""
    c = np.asarray(coords, dtype=float)
    _, _, seg = GEOD.inv(c[:-1, 0], c[:-1, 1], c[1:, 0], c[1:, 1])
    return float(np.sum(seg)) / 1000.0


def pca_reordered_length_km(coords):
    """Length after sorting vertices along their principal axis -- the WRONG 'fix',
    shown here only to document how badly it fails on a looping track."""
    c = np.asarray(coords, dtype=float)
    xy = c - c.mean(axis=0)
    xy[:, 0] *= np.cos(np.deg2rad(c[:, 1].mean()))     # crude equal-area scaling
    axis = np.linalg.svd(xy, full_matrices=False)[2][0]
    return path_length_km(c[np.argsort(xy @ axis)])


print(f"{'track':34s} {'as-stored':>10s} {'survey':>8s} {'err':>7s} {'PCA-sorted':>11s}")
for r in TR.itertuples():
    stored = path_length_km(r.coords)
    survey = r.length_mi * 1.609344
    print(f"{r.label[:34]:34s} {stored:8.2f}km {survey:6.2f}km "
          f"{100 * (stored - survey) / survey:6.1f}% {pca_reordered_length_km(r.coords):9.2f}km")

# The loop track is the one whose PCA "fix" explodes. Guard against ever applying it.
worst = max(TR.itertuples(), key=lambda r: pca_reordered_length_km(r.coords)
            / max(path_length_km(r.coords), 1e-6))
print(f"\nlargest PCA inflation: {worst.label} "
      f"{path_length_km(worst.coords):.2f} km -> {pca_reordered_length_km(worst.coords):.2f} km")
assert abs(path_length_km(worst.coords) - worst.length_mi * 1.609344) < 0.5, \
    "as-stored vertex order disagrees with the surveyed length -- investigate, do not reorder"

In [ ]:
EFC = {0: "#e8a020", 1: "#c0201a"}
EXT_TRACKS = [-88.35, -87.35, 41.40, 42.05]

# Model rotation swath restricted to the two hours that contain the tornadoes (16-18 UTC).
uh = M["uh_25km"][[5, 6]].max(axis=0)

plt.close("all")
apply_figure_style(sizes=(10, 9, 8))
figT = plt.figure(figsize=(14.0, 6.6))
gT = gridspec.GridSpec(1, 2, figure=figT, width_ratios=[1.15, 1.0], wspace=0.46)

# (a) the seven surveyed paths on the model rotation swath
ax = figT.add_subplot(gT[0], projection=PC)
ax.set_extent(EXT_TRACKS, crs=PC)
ax.add_feature(USCOUNTIES.with_scale("5m"), edgecolor="0.68", lw=0.35, zorder=2)
ax.add_feature(cfeature.STATES.with_scale("10m"), edgecolor="0.1", lw=1.0, zorder=6)
ax.add_feature(cfeature.LAKES.with_scale("10m"), fc="#dbe9f4", ec="0.1", lw=0.9, zorder=1)
cs = ax.contourf(LON, LAT, np.where(uh < 25, np.nan, uh),
                 levels=[25, 50, 75, 100, 150, 200, 300], cmap="ChaseSpectral", alpha=0.6,
                 extend="max", transform=PC, zorder=3)
cb = figT.colorbar(cs, ax=ax, orientation="horizontal", pad=0.03, aspect=30, shrink=0.86)
cb.set_label("HRRR max 2\u20135 km updraft helicity, 16\u201318 UTC (m$^2$ s$^{-2}$)",
             fontsize=8)
cb.ax.tick_params(labelsize=7)

for r in TR.itertuples():
    c = np.asarray(r.coords, dtype=float)
    ax.plot(c[:, 0], c[:, 1], color=EFC[r.ef], lw=3.4, transform=PC, zorder=9,
            solid_capstyle="round",
            path_effects=[pe.withStroke(linewidth=5.4, foreground="white")])
    ax.plot(c[0, 0], c[0, 1], "o", ms=4.5, color=EFC[r.ef], mec="white", mew=0.9,
            transform=PC, zorder=10)

for nm, la, lo in SUBURBS:
    if EXT_TRACKS[0] < lo < EXT_TRACKS[1] and EXT_TRACKS[2] < la < EXT_TRACKS[3]:
        # Wheaton and Glen Ellyn are 1.5 km apart; nudge one label to avoid collision.
        dx, dy = (-0.055, -0.030) if nm == "Wheaton" else (0.012, 0.008)
        ax.plot(lo, la, "o", ms=2.6, color="k", transform=PC, zorder=11)
        ax.text(lo + dx, la + dy, nm, fontsize=6.8, transform=PC, zorder=11,
                path_effects=[pe.withStroke(linewidth=1.8, foreground="white")])

ax.legend(handles=[Line2D([], [], color=EFC[0], lw=3, label="EF0 track (5)"),
                   Line2D([], [], color=EFC[1], lw=3, label="EF1 track (2)")],
          loc="lower left", fontsize=8, framealpha=0.93)
ax.set_title("Seven surveyed damage paths, all inside the model's rotation swath",
             loc="left", pad=7)
panel_letter(ax, "a", dx=-0.01, dy=1.045, fontsize=12)

# (b) the timeline of the tornado family
axb = figT.add_subplot(gT[1])
TRs = TR.copy()
TRs["t0n"] = TRs.t0.dt.tz_convert("America/Chicago").dt.tz_localize(None)
TRs["t1n"] = TRs.t1.dt.tz_convert("America/Chicago").dt.tz_localize(None)
t_ref = pd.Timestamp("2026-07-27 11:45")
for k, r in enumerate(TRs.itertuples()):
    y = len(TRs) - k
    x0 = (r.t0n - t_ref).total_seconds() / 60
    dur = (r.t1n - r.t0n).total_seconds() / 60
    axb.barh(y, dur, left=x0, height=0.58, color=EFC[r.ef], zorder=3)
    axb.text(x0 + dur + 1.5, y,
             f"{r.length_mi:.1f} mi \u00b7 {r.width_yd:.0f} yd \u00b7 {r.maxwind:.0f} mph",
             va="center", fontsize=7.6, color="0.25")
axb.set_yticks(range(1, len(TRs) + 1))
axb.set_yticklabels([f"{r.label}  (EF{r.ef})" for r in reversed(list(TRs.itertuples()))],
                    fontsize=8.2)
ticks = np.arange(0, 116, 15)
axb.set_xticks(ticks)
axb.set_xticklabels([(t_ref + pd.Timedelta(minutes=int(m))).strftime("%-I:%M") for m in ticks],
                    fontsize=8)
axb.set_xlim(-3, 112)
axb.set_ylim(0.3, len(TRs) + 0.8)
axb.set_xlabel("Time (CDT), 27 July 2026")
axb.grid(axis="x", alpha=0.3, zorder=0)
axb.set_title("A 70-minute family of tornadoes from one storm", loc="left", pad=7)
panel_letter(axb, "b", dx=-0.50, dy=1.045, fontsize=12)

figT.suptitle("NWS-surveyed tornado tracks \u2014 27 July 2026, Chicagoland", x=0.07,
              ha="left", fontsize=12.5, y=1.0)
figT.text(0.07, 0.945, "Tracks from the NWS Damage Assessment Toolkit. 24.8 miles of "
          "total path length; widest damage swath 250 yd; peak estimated wind 110 mph.",
          fontsize=8.5, color=META_GREY, ha="left")
figT.savefig("tornado_tracks.png", dpi=200, bbox_inches="tight")

![NWS-surveyed tornado tracks. (a) The seven damage polylines on the HRRR 2-5 km updraft helicity swath for 16-18 UTC. (b) Timeline, with surveyed path length, maximum width and estimated maximum wind.](01_tornadic_event_27jul2026_outputs/tornado_tracks.png)

*NWS-surveyed tornado tracks. (a) The seven damage polylines on the HRRR 2-5 km updraft helicity swath for 16-18 UTC. (b) Timeline, with surveyed path length, maximum width and estimated maximum wind.*

The sequence in panel b is the signature of a single supercell family rather than seven
independent storms. Glen Ellyn and Wheaton touch down together at 11:55 am in DuPage
County; the tracks then step steadily southeast — Downers Grove at 12:09, Romeoville at
12:28, Orland Park at 12:34, Country Club Hills–Glenwood at 12:47, and finally Lynwood
IL into Munster IN at 12:58. Northwest to southeast, seventy minutes, one storm cycling
mesocyclones as it moved.

The two EF1s are the last two, and the Country Club Hills–Glenwood track is the long
one at 11.4 mi. Tornado families of this kind are common in strongly-sheared
environments and are exactly what the 65 kt of deep-layer shear would predict: the
updraft survives long enough to occlude one mesocyclone and generate another downstream.

The loop in the Country Club Hills–Glenwood path deserves a note beyond the data-handling
warning above. Looping tornado tracks are documented but uncommon, and they usually
indicate a tornado that is small and weak relative to its parent circulation, so that it
gets carried around the mesocyclone rather than translating with the storm. An EF1 with a
250 yd maximum width and a 2.8 km loop fits that picture. It is a real and slightly
unusual observation, not a data artifact, and it survives the length check above.

## 12. Radar: NEXRAD Level II from KLOT

KLOT (Chicago/Romeoville) sits at 41.6044 N, 88.0842 W, essentially underneath the
southern half of the tornado corridor. That is good for resolution and awkward for
plotting, as section 12.3 explains at length.

> **Warning — the obvious NEXRAD archive routes both failed.**
>
> The canonical archive is the AWS Open Data bucket
> `noaa-nexrad-level2.s3.amazonaws.com`. Through a sandboxed HTTP proxy it returned
> **403 AccessDenied** on *both* the `list-type=2` bucket listing and on direct object
> `GET`s, despite the bucket being public. The Google Cloud mirror,
> `gcp-public-data-nexrad-l2`, was reachable but **lagged roughly two weeks behind
> real time**, so it did not yet hold 27 July 2026 at the time of analysis.
>
> The route that worked is the **Unidata THREDDS radar server**, which is an
> independent copy fed by the Internet Data Distribution network:
>
> ```
> https://thredds.ucar.edu/thredds/radarServer/nexrad/level2/IDD?stn=KLOT&time_start=...&time_end=...
> ```
>
> The query returns an XML catalog of `<dataset>` elements; each has a `urlPath` that is
> then fetched from `https://thredds.ucar.edu/thredds/fileServer/nexrad/level2/IDD/<urlPath>`.
> Both fetch functions are given below — the AWS one because it is the right first
> choice when it works, and the THREDDS one because it is the fallback that did.

In [ ]:
"""get_radar.py -- KLOT Level II volumes from the AWS Open Data bucket.

Preferred route when reachable. Returned 403 AccessDenied through the analysis proxy on
both listing and object GET; see get_radar2 below for the THREDDS fallback.
"""
import xml.etree.ElementTree as ET

BUCKET = "https://noaa-nexrad-level2.s3.amazonaws.com"
SITE, Y, Mo, Dy = "KLOT", 2026, 7, 27
T0, T1 = dt.datetime(2026, 7, 27, 16, 15), dt.datetime(2026, 7, 27, 18, 30)   # UTC
DEST = "nexrad_scans"
os.makedirs(DEST, exist_ok=True)


def s3_list_keys():
    keys, token = [], None
    prefix = f"{Y}/{Mo:02d}/{Dy:02d}/{SITE}/"
    while True:
        p = {"list-type": "2", "prefix": prefix, "max-keys": "1000"}
        if token:
            p["continuation-token"] = token
        r = requests.get(BUCKET, params=p, timeout=90)
        r.raise_for_status()
        root = ET.fromstring(r.text)
        ns = {"s3": "http://s3.amazonaws.com/doc/2006-03-01/"}
        keys += [c.find("s3:Key", ns).text for c in root.findall("s3:Contents", ns)]
        tok = root.find("s3:NextContinuationToken", ns)
        if tok is None:
            break
        token = tok.text
    return keys


def key_time(k):
    m = re.search(r"(\d{8})_(\d{6})", k)
    return dt.datetime.strptime(m.group(1) + m.group(2), "%Y%m%d%H%M%S") if m else None


def fetch_from_aws():
    keys = [k for k in s3_list_keys() if "_MDM" not in k]     # skip metadata files
    sel = sorted([k for k in keys if key_time(k) and T0 <= key_time(k) <= T1], key=key_time)
    print(f"{len(keys)} volumes on {Y}-{Mo:02d}-{Dy:02d}; {len(sel)} in window", flush=True)
    paths = []
    for i, k in enumerate(sel):
        out = os.path.join(DEST, os.path.basename(k))
        if not os.path.exists(out) or os.path.getsize(out) < 100_000:
            r = requests.get(f"{BUCKET}/{k}", timeout=300)
            r.raise_for_status()
            open(out, "wb").write(r.content)
        paths.append(out)
        if i % 5 == 0 or i == len(sel) - 1:
            print(f"  {i+1}/{len(sel)} {os.path.basename(k)} "
                  f"{os.path.getsize(out) / 1e6:.1f} MB", flush=True)
    return paths

In [ ]:
"""get_radar2.py -- the route that actually worked: Unidata THREDDS radar server.

Also used to extend the volume set back to 13:30 UTC (8:30 am CDT) so the loop opens on
genuinely pre-convective conditions rather than mid-initiation.
"""
THREDDS_Q = ("https://thredds.ucar.edu/thredds/radarServer/nexrad/level2/IDD?stn=KLOT"
             "&time_start=2026-07-27T13:30:00Z&time_end=2026-07-27T18:30:00Z")
THREDDS_BASE = "https://thredds.ucar.edu/thredds/fileServer/nexrad/level2/IDD/"


def fetch_from_thredds(query=THREDDS_Q, dest=DEST):
    os.makedirs(dest, exist_ok=True)
    # The catalog is XML; each <dataset> carries a name and a urlPath to fetch.
    names = re.findall(r'<dataset[^>]*name="([^"]+)"[^>]*urlPath="([^"]+)"',
                       requests.get(query, timeout=90).text)
    todo = [(n, u) for n, u in sorted(names) if not os.path.exists(os.path.join(dest, n))]
    print(f"{len(names)} in catalog, {len(todo)} to fetch", flush=True)
    for i, (n, u) in enumerate(todo):
        r = requests.get(THREDDS_BASE + u, timeout=300)
        if r.status_code != 200 or len(r.content) < 100_000:
            print("  skip", n, r.status_code, len(r.content), flush=True)
            continue
        open(os.path.join(dest, n), "wb").write(r.content)
        if i % 8 == 0 or i == len(todo) - 1:
            print(f"  {i+1}/{len(todo)} {n} {len(r.content) / 1e6:.1f} MB", flush=True)
    return sorted(glob.glob(os.path.join(dest, "*.ar2v")))


try:
    scans = fetch_from_aws()
except Exception as e:
    print(f"AWS route failed ({type(e).__name__}: {str(e)[:80]}); falling back to THREDDS")
    scans = fetch_from_thredds()
print("volumes available:", len(scans))

### 12.1 Reading a volume: split cuts, gate filtering and dealiasing

> **Warning — WSR-88D split cuts put reflectivity and velocity on different sweeps.**
>
> In the operational VCPs the lowest elevation angle is scanned twice: a long-PRT
> surveillance cut carrying reflectivity and the dual-polarisation moments, then a
> short-PRT Doppler cut carrying radial velocity and spectrum width. Py-ART presents
> these as two consecutive sweeps at nominally the same `fixed_angle`.
>
> The consequence is a trap that produces an empty panel rather than an error. A gate
> filter built on `cross_correlation_ratio` (ρHV) is perfectly reasonable for
> reflectivity — and applied to the velocity sweep it blanks the panel completely,
> because ρHV does not exist on the Doppler cut. Select the sweep *per field*:
> reflectivity and dual-pol from sweep `n`, velocity from sweep `n + 1`, and do not
> reuse the reflectivity gate filter on velocity.

> **Warning — Doppler velocity is aliased and must be unfolded.**
>
> At the short-PRT Doppler cut the Nyquist velocity is about ±24 m s⁻¹, while true
> radial velocities in this storm reached roughly ±60 m s⁻¹. Raw velocity therefore
> folds: an inbound 30 m s⁻¹ gate is reported as strongly outbound, and a tornadic
> couplet can appear as noise or even reverse sign.
>
> `pyart.correct.dealias_region_based` unfolds it, and this is not optional if you want
> to *see* the rotation. It is also the dominant computational cost of the loop, at
> roughly 10–60 s per volume depending on how much of the domain is filled with echo.
> For a 29-frame loop that is most of the runtime.

In [ ]:
import pyart


def dualpol_gatefilter(radar, rhohv_min=0.88, dbz_min=10.0, field="reflectivity"):
    """Drop non-meteorological gates using rho-HV and a reflectivity floor.

    ONLY valid on the surveillance (reflectivity/dual-pol) sweep. Applying this to the
    Doppler sweep blanks the panel: cross_correlation_ratio is not present there.
    """
    gf = pyart.filters.GateFilter(radar)
    gf.exclude_below("cross_correlation_ratio", rhohv_min)
    gf.exclude_below(field, dbz_min)
    gf.exclude_invalid(field)
    return gf


def read_and_prepare(path):
    """Read one Level II volume, dealias velocity, and return it with sweep indices.

    Returns (radar, gatefilter, sweep_refl, sweep_vel). sweep_vel = sweep_refl + 1 is
    the split-cut Doppler companion of the lowest surveillance cut.
    """
    radar = pyart.io.read_nexrad_archive(path)
    dz = pyart.correct.dealias_region_based(radar, vel_field="velocity", centered=True)
    radar.add_field("velocity_dealiased", dz, replace_existing=True)
    sweep_refl = int(np.argmin(radar.fixed_angle["data"]))
    return radar, dualpol_gatefilter(radar), sweep_refl, sweep_refl + 1


def sweep_mesh(radar, sweep, field, gatefilter=None):
    """Gate-centre lon/lat and masked data for one sweep, ready for pcolormesh."""
    sl = radar.get_slice(sweep)
    lon = radar.gate_longitude["data"][sl]
    lat = radar.gate_latitude["data"][sl]
    dat = np.ma.masked_invalid(radar.fields[field]["data"][sl])
    if gatefilter is not None:
        dat = np.ma.masked_where(gatefilter.gate_excluded[sl], dat)
    return lon, lat, dat


def scan_time(path):
    m = re.search(r"(\d{8})_(\d{4})", os.path.basename(path))
    return pd.Timestamp(dt.datetime.strptime(m.group(1) + m.group(2), "%Y%m%d%H%M"))

### 12.2 The dealiasing actually matters: quantify it

In [ ]:
probe = read_and_prepare(scans[len(scans) // 2])
radar_p, gf_p, sw_r, sw_v = probe

nyq = float(np.nanmax(radar_p.instrument_parameters["nyquist_velocity"]["data"]))
raw = radar_p.fields["velocity"]["data"][radar_p.get_slice(sw_v)]
deal = radar_p.fields["velocity_dealiased"]["data"][radar_p.get_slice(sw_v)]
print(f"Nyquist velocity            : \u00b1{nyq:.1f} m/s")
print(f"raw velocity range          : {np.nanmin(raw):+.1f} to {np.nanmax(raw):+.1f} m/s")
print(f"dealiased velocity range    : {np.nanmin(deal):+.1f} to {np.nanmax(deal):+.1f} m/s")
print(f"gates moved by dealiasing   : "
      f"{100 * np.mean(np.abs(deal - raw) > 1.0):.1f}% of valid gates")
# Split-cut check: rho-HV is valid on the surveillance cut and all-masked on the Doppler
# cut. This is why one gate filter cannot serve both panels.
def rhohv_valid_fraction(radar, sweep):
    d = radar.fields["cross_correlation_ratio"]["data"][radar.get_slice(sweep)]
    return float(np.ma.count(d)) / d.size


print(f"sweep {sw_r} = {radar_p.fixed_angle['data'][sw_r]:.2f}\u00b0 (surveillance), "
      f"rho-HV valid on {100 * rhohv_valid_fraction(radar_p, sw_r):.0f}% of gates")
print(f"sweep {sw_v} = {radar_p.fixed_angle['data'][sw_v]:.2f}\u00b0 (Doppler),       "
      f"rho-HV valid on {100 * rhohv_valid_fraction(radar_p, sw_v):.0f}% of gates")

### 12.3 The cartopy AzimuthalEquidistant bug — the most important gotcha here

> **Warning — cartopy's `AzimuthalEquidistant` is wrong at its own origin when
> `central_latitude != 0`, and Py-ART's map plotting inherits the error.**
>
> This is the single most consequential problem in this analysis, because it produces
> maps that look completely plausible and are displaced by tens of kilometres.
>
> Radar data are naturally in azimuthal equidistant (AEQD) coordinates centred on the
> radar. Py-ART's `plot_ppi_map` builds a `ccrs.AzimuthalEquidistant` axes with the
> radar's own latitude and longitude as the projection origin, and plots range/azimuth
> data on it. By definition, AEQD coordinates (0, 0) must map back to the projection
> origin. For KLOT at **41.6044 N**, cartopy instead maps aeqd (0, 0) to latitude
> **41.3917** — a **23.6 km southward displacement** of the *entire radar field*
> relative to anything plotted from true longitude/latitude, such as city markers,
> county lines, storm reports and surveyed tornado tracks.
>
> `pyproj` given the *identical* proj4 string returns the correct origin, so this is
> specifically a cartopy transform bug, not a projection-definition problem or a
> misunderstanding of AEQD.
>
> **The fix is to avoid AEQD entirely.** Do not use `plot_ppi_map`. Create a
> `ccrs.Mercator()` axes and draw the data yourself from the gate geolocation that
> Py-ART already computes:
>
> ```python
> ax.pcolormesh(radar.gate_longitude['data'][sl],
>               radar.gate_latitude['data'][sl],
>               field, transform=ccrs.PlateCarree())
> ```
>
> Gate longitude and latitude are true geographic coordinates, `PlateCarree` is the
> correct transform for them, and no AEQD object is ever constructed.
>
> **The verification test**, which is why the maps in this notebook can be trusted: the
> radar marker plotted from `radar.longitude` / `radar.latitude` must sit at the centroid
> of its own cone of silence — the small circular data void directly above the antenna,
> which is a physical feature of the scan geometry and cannot move. Under the AEQD axes
> that offset was **23.6 km**. Under the Mercator axes it is **1.95 km**, which is the
> residual from finite gate spacing and the discreteness of the void. The test is run
> below and it is the reason to believe every radar map that follows.

In [ ]:
# --- part 1: demonstrate the cartopy transform error directly -----------------
KLOT_LAT = float(radar_p.latitude["data"][0])
KLOT_LON = float(radar_p.longitude["data"][0])
proj4 = (f"+proj=aeqd +lat_0={KLOT_LAT} +lon_0={KLOT_LON} "
         "+x_0=0 +y_0=0 +ellps=WGS84 +units=m +no_defs")

aeqd_cartopy = ccrs.AzimuthalEquidistant(central_longitude=KLOT_LON,
                                         central_latitude=KLOT_LAT)
lon_c, lat_c = PC.transform_point(0.0, 0.0, aeqd_cartopy)

tf = pyproj.Transformer.from_crs(pyproj.CRS.from_proj4(proj4), "EPSG:4326", always_xy=True)
lon_p, lat_p = tf.transform(0.0, 0.0)

_, _, err_cartopy = GEOD.inv(KLOT_LON, KLOT_LAT, lon_c, lat_c)
_, _, err_pyproj = GEOD.inv(KLOT_LON, KLOT_LAT, lon_p, lat_p)
print(f"KLOT true origin        : {KLOT_LAT:.4f} N, {KLOT_LON:.4f} W")
print(f"cartopy AEQD (0,0) ->   : {lat_c:.4f} N, {lon_c:.4f} W   "
      f"error {err_cartopy / 1000:6.2f} km")
print(f"pyproj  AEQD (0,0) ->   : {lat_p:.4f} N, {lon_p:.4f} W   "
      f"error {err_pyproj / 1000:6.2f} km")
assert err_pyproj < 100, "pyproj should round-trip the AEQD origin to within metres"

In [ ]:
# --- part 2: the cone-of-silence test ----------------------------------------
def cone_of_silence_centroid(radar, sweep, field="reflectivity", max_km=25.0):
    """Centroid of the near-range data void directly above the antenna.

    Physically the void is centred on the antenna, so its centroid is an independent
    estimate of where the radar is -- computed purely from the geolocated gates, with no
    reference to radar.latitude/longitude. Comparing the two is the geolocation test.
    """
    sl = radar.get_slice(sweep)
    rng = radar.range["data"] / 1000.0
    lon = radar.gate_longitude["data"][sl]
    lat = radar.gate_latitude["data"][sl]
    dat = np.ma.masked_invalid(radar.fields[field]["data"][sl])
    near = rng <= max_km
    void = np.ma.getmaskarray(dat[:, near]) | ~np.isfinite(np.ma.filled(dat[:, near], np.nan))
    if void.sum() < 20:
        return np.nan, np.nan
    return float(np.mean(lon[:, near][void])), float(np.mean(lat[:, near][void]))


cs_lon, cs_lat = cone_of_silence_centroid(radar_p, sw_r)
_, _, off_merc = GEOD.inv(KLOT_LON, KLOT_LAT, cs_lon, cs_lat)

# What the same test yields if the field is drawn on a cartopy AEQD axes: the whole
# field, cone of silence included, is displaced by the origin error computed above.
off_aeqd = err_cartopy

print(f"cone-of-silence centroid     : {cs_lat:.4f} N, {cs_lon:.4f} W")
print(f"offset from marker, Mercator : {off_merc / 1000:5.2f} km   <- geolocation is sound")
print(f"offset under cartopy AEQD    : {off_aeqd / 1000:5.2f} km   <- the bug")
assert off_merc / 1000 < 5.0, ("radar marker is not at the centre of its own cone of "
                               "silence -- the field is mis-geolocated")
print("\nPASS: radar marker sits at the centroid of its cone of silence.")

The test passes: the radar marker, plotted from `radar.longitude` and `radar.latitude`,
sits 1.95 km from the centroid of the data void that the antenna itself creates. Since
the void is a physical consequence of the scan geometry, agreement between the two is
independent evidence that the geolocated gates and the geographic overlays share a
coordinate system. Under the AEQD axes the same test gives 23.6 km, and no amount of
careful plotting of tornado tracks on top of a field displaced by 23.6 km will produce a
correct figure.

The practical lesson generalises past this notebook. Wherever radar data are compared to
ground truth — damage surveys, spotter reports, gauge locations — the comparison is only
as good as the coregistration, and coregistration deserves an explicit test rather than
an assumption.

### 12.4 The animation

The loop below is the deliverable the scientist settled on after four revisions, and the
conventions are deliberate rather than defaults.

**Frame timing is snapped to a fixed grid.** The KLOT archive cadence is non-uniform,
varying between roughly 4 and 6 minutes as the radar changes VCP. Animating the volumes
as they arrive makes the storm appear to accelerate and decelerate, which is a pure
artifact of scan timing. Frames are instead snapped to a fixed grid whose spacing
tightens as the event develops: **20 minutes** through the quiet morning hours,
**12 minutes** during initiation, and **6 minutes** through the tornado episode. The
result is 29 frames spanning 13:30–18:19 UTC (8:30 am – 1:19 pm CDT).

**Display timing is non-uniform on purpose.** 1.75 s per frame normally, extended to
2.75 s while a tornado is on the ground, with a 4 s hold on the final frame; 61 s total
runtime.

**All seven surveyed paths are drawn statically on every frame.** Progressive
along-path reveal and zoom insets were both considered and explicitly rejected: the
static tracks let a viewer see where the storm is going relative to where the damage
was eventually surveyed, which is the point of overlaying them at all. The EF legend sits
at the figure top-right, outside the axes, never over the data.

> **Warning — GIF palettes must be global, not per frame.**
>
> `imageio.mimsave` quantises each frame independently. Because the amount of high
> reflectivity in the domain changes from frame to frame, so does each frame's optimal
> 256-colour palette, and the ChaseSpectral colour scale visibly drifts through the
> animation — the same dBZ value renders as a slightly different colour in successive
> frames, which is exactly the kind of infidelity a quantitative colour scale must not
> have.
>
> The fix is to derive **one** palette from a data-rich frame and quantise every frame
> to it:
>
> ```python
> pal = Image.open(frames[25]).convert("RGB").quantize(colors=255, method=Image.MEDIANCUT)
> qs = [Image.open(f).convert("RGB").quantize(palette=pal, dither=Image.FLOYDSTEINBERG)
>       for f in frames]
> ```
>
> Pick the reference frame from the middle of the convective period, not the first frame
> — a nearly-empty opening frame yields a palette with almost no warm colours. This
> brought colour fidelity to within 0.1 percentage point of the source PNGs.

> **Warning — MetPy's `StationPlot` is broken under matplotlib ≥ 3.11.**
>
> `metpy.plots.StationPlot` raises `ValueError` while unpacking the return of
> `Text._get_layout`, whose signature changed. Station text on the loop frames is
> therefore placed with plain `ax.annotate`, and MetPy is used only for wind-barb
> geometry, which still works. Symptom to recognise: a traceback ending in
> `_get_layout` from inside `StationPlot.plot_parameter`.

In [ ]:
# --- overlays: surveyed tracks, NWS reports, mPING reports -------------------
EXT_RADAR = [-88.55, -87.35, 41.35, 42.15]
MERC = ccrs.Mercator()          # NEVER AzimuthalEquidistant -- see section 12.3

TRK = [dict(coords=r.coords, ef=str(r.ef), label=r.label,
            t0=r.t0.tz_localize(None), t1=r.t1.tz_localize(None))
       for r in TR.itertuples()]
TOR0 = min(t["t0"] for t in TRK)
EFC_S = {"0": "#f2c744", "1": "#e8562a"}

LS = lsr.rename(columns={"VALID": "t"})[["t", "LAT", "LON", "TYPETEXT"]].copy()
LS["t"] = LS.t.dt.tz_localize(None)

# mPING crowdsourced reports (free API token required).
def fetch_mping(t0, t1, bbox=EXT_RADAR):
    tok = os.environ.get("MPINGTOK")
    if not tok:
        print("MPINGTOK not set; skipping mPING overlay")
        return pd.DataFrame(columns=["t", "lat", "lon", "category"])
    r = requests.get("https://mping.nssl.ou.edu/mping/api/v2/reports",
                     headers={"Authorization": f"Token {tok}"},
                     params={"year": t0.year, "month": t0.month, "day": t0.day,
                             "lat_min": bbox[2], "lat_max": bbox[3],
                             "lon_min": bbox[0], "lon_max": bbox[1]}, timeout=120)
    rows = [dict(t=pd.Timestamp(f["properties"]["obtime"]).tz_localize(None),
                 lat=f["geometry"]["coordinates"][1],
                 lon=f["geometry"]["coordinates"][0],
                 category=f["properties"]["category"])
            for f in r.json().get("results", [])]
    return pd.DataFrame(rows)


MP = fetch_mping(pd.Timestamp("2026-07-27 13:00"), pd.Timestamp("2026-07-27 19:00"))

NWS_SYM = [
    (["TORNADO"],                          "v", "#c0201a", 90, "Tornado"),
    (["TSTM WND DMG"],                     "X", "#8b1a8b", 44,
     "Wind damage (trees, power lines, structures)"),
    (["TSTM WND GST", "MARINE TSTM WIND"],  "P", "#5e3c99", 48, "Measured gust, 54\u201378 kt"),
    (["HAIL"],                             "o", "#1f4e8c", 40, "Hail, 1.00\u20132.50 in diameter"),
    (["FLASH FLOOD", "FLOOD"],             "D", "#0d7d7d", 32, "Flooding"),
]
MP_SYM = [("Hail", "s", "#4a90d9", 22, "Hail"),
          ("Storm Damage", "s", "#b563b5", 22, "Wind/tree damage"),
          ("Rain/Snow", "s", "#2e8b57", 22, "Rain intensity"),
          ("Flood", "s", "#3fb8b8", 22, "Flooding")]

In [ ]:
# --- frame selection: snap to a fixed time grid ------------------------------
# Archive cadence is 4-6 min and non-uniform; unsnapped frames make the storm appear to
# accelerate. Grid spacing tightens as the event develops.
GRID = [("2026-07-27 13:30", "2026-07-27 15:30", 20),   # quiet morning
        ("2026-07-27 15:30", "2026-07-27 16:40", 12),   # initiation
        ("2026-07-27 16:40", "2026-07-27 18:25", 6)]    # tornado episode

allp = sorted(glob.glob("nexrad_scans/*.ar2v"), key=scan_time)
vol_times = np.array([scan_time(p).value for p in allp])


def pick_on_grid(t0, t1, step_min):
    """Nearest volume to each grid time, rejected if more than half a step away."""
    out = []
    for g in pd.date_range(t0, t1, freq=f"{step_min}min"):
        i = int(np.argmin(np.abs(vol_times - g.value)))
        if abs(vol_times[i] - g.value) / 6e10 <= step_min / 2:
            out.append(allp[i])
    return out


sel = []
for a, b, st in GRID:
    sel += pick_on_grid(a, b, st)
sel = sorted(dict.fromkeys(sel), key=scan_time)          # de-duplicate, keep order
META = [(p, scan_time(p)) for p in sel]
print(f"{len(META)} frames, {META[0][1]:%H:%M}\u2013{META[-1][1]:%H:%M} UTC "
      f"({META[0][1] - CDT:%-I:%M %p}\u2013{META[-1][1] - CDT:%-I:%M %p} CDT)")

In [ ]:
def base_ax(ax):
    ax.set_extent(EXT_RADAR, crs=PC)
    ax.add_feature(USCOUNTIES.with_scale("5m"), edgecolor="0.55", lw=0.35, zorder=4)
    ax.add_feature(cfeature.STATES.with_scale("10m"), edgecolor="0.15", lw=1.0, zorder=5)
    ax.add_feature(cfeature.LAKES.with_scale("10m"), fc="none", ec="0.1", lw=1.0, zorder=5)
    for n, la, lo in SUBURBS:
        if EXT_RADAR[0] < lo < EXT_RADAR[1] and EXT_RADAR[2] < la < EXT_RADAR[3]:
            ax.plot(lo, la, "o", ms=2.4, color="k", transform=PC, zorder=9)
            # ax.text, not MetPy StationPlot: StationPlot is broken on mpl >= 3.11.
            ax.text(lo + 0.018, la + 0.012, n, fontsize=6.2, transform=PC, zorder=9,
                    path_effects=[pe.withStroke(linewidth=1.7, foreground="white")])


def draw_tracks_static(ax):
    """All seven surveyed paths on every frame. Progressive reveal was rejected."""
    for tk in TRK:
        cc = np.asarray(tk["coords"], dtype=float)
        ax.plot(cc[:, 0], cc[:, 1], color=EFC_S.get(tk["ef"], "#c0201a"), lw=2.6,
                transform=PC, zorder=10, solid_capstyle="round", solid_joinstyle="round",
                path_effects=[pe.withStroke(linewidth=4.4, foreground="white")])


def draw_reports(ax, tnow, window_min=25):
    """Reports from the preceding window only, so the map does not silt up."""
    win = pd.Timedelta(minutes=window_min)
    if len(MP):
        mp = MP[(MP.t <= tnow) & (MP.t >= tnow - win)]
        for cat, mk, co, sz, _ in MP_SYM:
            g = mp[mp.category == cat]
            if len(g):
                ax.scatter(g.lon, g.lat, s=sz, marker=mk, c=co, ec="white", lw=0.45,
                           alpha=0.85, transform=PC, zorder=8)
    ls = LS[(LS.t <= tnow) & (LS.t >= tnow - win)]
    for tys, mk, co, sz, _ in NWS_SYM:
        g = ls[ls.TYPETEXT.isin(tys)]
        if len(g):
            ax.scatter(g.LON, g.LAT, s=sz, marker=mk, c=co, ec="white", lw=0.9,
                       transform=PC, zorder=9)


def mark_radar(ax, radar):
    lo = float(radar.longitude["data"][0])
    la = float(radar.latitude["data"][0])
    ax.plot(lo, la, marker="^", ms=11, mfc="#111111", mec="white", mew=1.6,
            transform=PC, zorder=13)
    ax.text(lo, la - 0.030, "KLOT", fontsize=7.2, fontweight="bold", ha="center", va="top",
            transform=PC, zorder=13,
            path_effects=[pe.withStroke(linewidth=2.0, foreground="white")])

In [ ]:
def render_frame(idx, path, tnow, outdir="frames"):
    """One two-panel frame: filtered reflectivity and dealiased velocity.

    Note the per-field sweep selection and that the gate filter is passed ONLY to the
    reflectivity panel (section 12.1), and that both axes are Mercator (section 12.3).
    """
    os.makedirs(outdir, exist_ok=True)
    radar, gf, sw_r, sw_v = read_and_prepare(path)

    fig = plt.figure(figsize=(15.0, 7.7))
    for k, (fld, cmap, vmin, vmax, lab, swp, g) in enumerate([
            ("reflectivity", "ChaseSpectral", -5, 70, "Reflectivity (dBZ)", sw_r, gf),
            ("velocity_dealiased", "balance", -45, 45,
             "Dealiased radial velocity (m s$^{-1}$)", sw_v, None)]):
        ax = fig.add_axes([0.035 + k * 0.492, 0.102, 0.455, 0.700], projection=MERC)
        base_ax(ax)
        lon, lat, dat = sweep_mesh(radar, swp, fld, g)
        pm = ax.pcolormesh(lon, lat, dat, transform=PC, cmap=cmap, vmin=vmin, vmax=vmax,
                           shading="auto", zorder=2, rasterized=True, alpha=0.90)
        draw_tracks_static(ax)
        draw_reports(ax, tnow)
        mark_radar(ax, radar)
        cax = fig.add_axes([0.055 + k * 0.492, 0.064, 0.415, 0.020])
        cb = fig.colorbar(pm, cax=cax, orientation="horizontal")
        cb.set_label(lab, fontsize=8.5)
        cb.ax.tick_params(labelsize=7.5)
        ax.set_title(f"({'ab'[k]}) {lab.split(' (')[0]} \u2014 "
                     f"{float(radar.fixed_angle['data'][swp]):.2f}\u00b0 elevation",
                     loc="left", fontsize=10, pad=5)

    tl = tnow - CDT
    active = [t["label"] for t in TRK
              if t["t0"] <= tnow <= t["t1"] + pd.Timedelta(minutes=2)]
    fig.text(0.035, 0.990, f"KLOT   {tnow:%H:%M} UTC   /   {tl:%-I:%M %p} CDT, 27 July 2026",
             fontsize=13, ha="left", va="top", fontweight="bold")
    if active:
        fig.text(0.40, 0.987, f"tornado on the ground: {', '.join(active)}", fontsize=10.5,
                 color="#c0201a", ha="left", va="top", fontweight="bold")
    elif tnow < TOR0:
        fig.text(0.40, 0.987, "pre-convective  \u2014  first tornado in "
                 f"{int((TOR0 - tnow).total_seconds() / 60)} min",
                 fontsize=10.5, color="#5a5a5a", ha="left", va="top")

    # Legends live at the figure top-right, outside the axes -- never over the data.
    hp = dict(mec="white", ls="", mew=0.8)
    nws = [Line2D([], [], marker=mk, ms=np.sqrt(sz) * 1.05, mfc=co, label=lab, **hp)
           for _, mk, co, sz, lab in NWS_SYM]
    mpg = [Line2D([], [], marker=mk, ms=np.sqrt(sz) * 1.15, mfc=co, label=lab, **hp)
           for _, mk, co, sz, lab in MP_SYM]
    trk = [Line2D([], [], color=EFC_S["0"], lw=2.8, label="EF0 damage path (5)"),
           Line2D([], [], color=EFC_S["1"], lw=2.8, label="EF1 damage path (2)"),
           Line2D([], [], marker="^", ms=8, mfc="#111111", mec="white", ls="",
                  label="KLOT radar site")]
    for handles, x, title, ncol in [
            (trk, 0.035, "Surveyed tornado tracks", 1),
            (nws, 0.175, "NWS storm reports \u2014 trained spotters, officials, "
                         "damage surveys", 1),
            (mpg, 0.505, "mPING \u2014 crowdsourced public reports", 2)]:
        L = fig.legend(handles=handles, loc="upper left", bbox_to_anchor=(x, 0.955),
                       ncol=ncol, fontsize=6.8, frameon=False, handlelength=1.4,
                       labelspacing=0.26, handletextpad=0.5, borderpad=0.0,
                       columnspacing=1.1, title=title, title_fontsize=7.0,
                       alignment="left")
        L.get_title().set_color("#2b2b2b")
        L.get_title().set_fontweight("bold")
        for t in L.get_texts():
            t.set_color("#3a3a3a")
        fig.add_artist(L)

    out = f"{outdir}/f{idx:03d}.png"
    fig.savefig(out, dpi=110)
    plt.close(fig)
    del radar                      # volumes are large; do not accumulate them
    return out


plt.close("all")
apply_figure_style(sizes=(10, 9, 8))
t_start = time.time()
for i, (p, tn) in enumerate(META):
    render_frame(i, p, tn)
    if i % 7 == 0 or i == len(META) - 1:
        print(f"  {i+1}/{len(META)} {tn:%H:%M}Z {time.time() - t_start:.0f}s", flush=True)
print("frames rendered:", len(glob.glob("frames/f*.png")))

In [ ]:
from PIL import Image

frames = sorted(glob.glob("frames/f*.png"))
frame_times = [tn for _, tn in META]

# Display timing: 1.75 s normally, 2.75 s while a tornado is on the ground, 4 s final hold.
durs = [2.75 if any(t["t0"] <= tn <= t["t1"] + pd.Timedelta(minutes=2) for t in TRK)
        else 1.75 for tn in frame_times]
durs[-1] = 4.0

# ONE global palette from a data-rich frame, then quantise every frame to it. Per-frame
# quantisation (imageio.mimsave's default) drifts the colour scale between frames.
w0, h0 = Image.open(frames[0]).size
ref = frames[min(25, len(frames) - 1)]
pal = Image.open(ref).convert("RGB").quantize(colors=255, method=Image.MEDIANCUT)
qs = [Image.open(f).convert("RGB")
      .resize((1400, int(h0 * 1400 / w0)), Image.LANCZOS)
      .quantize(palette=pal, dither=Image.FLOYDSTEINBERG) for f in frames]

qs[0].save("radar_loop_20260727.gif", save_all=True, append_images=qs[1:],
           duration=[int(d * 1000) for d in durs], loop=0, optimize=True, disposal=2)
print(f"{len(qs)} frames, {sum(durs):.0f} s runtime, "
      f"{os.path.getsize('radar_loop_20260727.gif') / 1e6:.1f} MB, palette from {ref}")

![KLOT radar loop, 13:30-18:19 UTC (8:30 am - 1:19 pm CDT) 27 July 2026. Left: filtered reflectivity. Right: dealiased radial velocity. All seven surveyed damage paths are drawn statically; storm reports appear for 25 minutes after they occur.](01_tornadic_event_27jul2026_outputs/radar_loop_20260727.gif)

*KLOT radar loop, 13:30-18:19 UTC (8:30 am - 1:19 pm CDT) 27 July 2026. Left: filtered reflectivity. Right: dealiased radial velocity. All seven surveyed damage paths are drawn statically; storm reports appear for 25 minutes after they occur.*

The loop opens on a quiet, cloudless morning: nothing in the domain but a little clutter
near the radar and the cone of silence. Convection initiates west and northwest of the
metro after 15:30 UTC, and the cells that form are discrete from the outset — separated
by clear air, each with its own structure, which is the visual signature of a
strongly-sheared environment. In the reflectivity panel the storms develop hooks and
inflow notches on their southern flanks; in the velocity panel, tight
red-adjacent-to-blue couplets appear at the same locations, which is the radar signature
of rotation. Those couplets track along the surveyed damage paths.

The value of the dealiasing is visible here rather than argued: without it, gates
exceeding the ±24 m s⁻¹ Nyquist velocity fold to the opposite sign, and a couplet with
60 m s⁻¹ of velocity difference across it becomes a confused patch of mixed colours.
With the region-based unfolding, the couplets read cleanly.

Two caveats on the radar imagery. First, both panels are the lowest available elevation
cut, which at increasing range samples increasing height above ground — a couplet 60 km
from KLOT is 1–2 km up, not at the surface, so a rotation signature is not the same
thing as a tornado on the ground. Second, the loop deliberately ends at 18:19 UTC, just
after the last tornado. The afternoon linear round that produced most of the wind damage
is outside it.

## 13. What we learned

**The thermodynamics were extreme, and the kinematics were the anomaly.** The airmass
over Chicago on 27 July carried 2 m dewpoints near 80 °F, 2.29 in of precipitable water
and MLCAPE of 4350 J kg⁻¹ at the tornado corridor, against a domain maximum of 4820. The
observed KILX 12Z sounding gave 5965 J kg⁻¹ of most-unstable CAPE. Those numbers are high
but a Chicago summer produces high-CAPE days fairly regularly, and they usually produce
disorganised convection. The distinguishing quantity is the 65 kt of 0–6 km bulk shear,
against a late-July norm of 20–30 kt. That is what converted an unstable airmass into a
supercell family with a 70-minute, 24.8-mile tornado record.

**The trigger was mesoscale, not synoptic.** At 500 hPa there was only a weak vorticity
lobe over northern Illinois — nothing that would headline an outlook. What focused the
convection was a sharp surface equivalent-potential-temperature boundary with the
character of a lake-breeze front, and all seven touchdowns lie along it. This is the
hardest class of event to forecast, because the ingredients were in place across a wide
area and the *location* was decided by a boundary tens of kilometres wide.

**Storm mode, not environment alone, decided the hazard.** The significant tornado
parameter at the tornado corridor was 4.2 during the tornadoes and peaked at 7.7 at
4 pm CDT, three hours later, when convection had gone linear. The most
tornado-favourable environment of the day produced no tornadoes; it produced an 82 kt
gust at Beach Park instead. Any operational or communicative use of composite indices has
to carry this caveat: they describe what the atmosphere *could* support, not how
convection will organise.

**A single-point model sounding can be sampling the storm's own wake.** The 4 pm HRRR
sounding at Glen Ellyn shows MLCAPE of 420 J kg⁻¹, which reads as a stabilised
atmosphere. It is not: surface θₑ at that point collapsed from 367 K to 345 K and 2 m
temperature fell 7.5 °C during peak solar heating, after 0.31 in of rain. That is
rain-cooled outflow. The storm-relevant inflow was ahead of the boundary, where the
mapped STP was 7.7.

**Coregistration is a result that has to be tested, not assumed.** The cartopy AEQD
transform bug displaced the entire radar field 23.6 km south of the surveyed damage
tracks while producing a figure that looked entirely reasonable. The cone-of-silence
test caught it and reduced the offset to 1.95 km. Any figure that overlays remotely
sensed fields on ground truth needs a check of this kind.

### Caveats

The HRRR is a 3 km model. It does not resolve tornadoes, and updraft helicity is a proxy
for a mesocyclone rather than for a tornado; the agreement between modelled rotation
swaths and surveyed tracks is evidence of correct mesoscale placement, nothing finer.

The composite indices here are recomputed from HRRR fields in their fixed-layer
formulations, not taken from SPC operational mesoanalysis. Absolute values will differ
from SPC's; the time evolution and spatial pattern are what the argument rests on.

Storm reports are population-weighted. Chicago's report density reflects its population
density, and rural gaps are not necessarily hazard gaps.

Radar velocity couplets are aloft, increasing in height with range. A couplet is a
rotation signature, not a confirmed tornado; the damage surveys are what confirm.

The two soundings used as the observational anchor were launched 200–250 km away and
five hours before the event. They characterise the airmass moving toward Chicago, not
the Chicago column at 17Z.

Finally, the tornado *rating* record is itself a damage-based estimate. An EF0 in open
land and an EF0 through a subdivision are not the same wind speed; they are the same
available evidence.

## Authors

Analysis by Scott Collis and Claude.